# pSMAD_2024-08-21 — 03_gradient_quantification

**Feeds:** ED Fig 2j, 2k

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# pSMAD Gradient Quantification

## Notebook Scope

This notebook quantifies shell-based pSMAD response for dataset 2 using the stitched scene images, cyst ROIs, and manual bead annotations. For multi-bead scenes, each cyst is assigned one primary bead by minimum bead-to-cyst-boundary distance, then the downstream distance and orientation calculations follow the original pSMAD workflow unchanged. Ambiguous primary-bead assignments are recorded and flagged here for later review in a follow-up notebook, but they are not reclassified or filtered in this notebook.

**Geometry Definitions Used Throughout**

For each bead-present cyst, geometry is defined from the cyst mask and the cyst-array center:
- major axis from PCA on mask pixels,
- boundary endpoints from axis projection,
- array center = center-cyst centroid when available, otherwise the mean centroid of cysts in that scene,
- anterior midpoint = axis endpoint closer to the array center,
- posterior midpoint = axis endpoint farther from the array center,
- `d` = distance from bead centroid to posterior midpoint,
- posterior crescent = posterior-facing band extending inward from the cyst edge,
- `x` = signed lateral coordinate on posterior-crescent pixels,
- `theta_b` = signed angle between the anterior→posterior midline and the anterior→bead vector,
- `theta` = signed angle on posterior-crescent pixels relative to the midline.

Center cysts are excluded from bead-present quantification.


## Setup


### Imports

Load the packages used throughout the notebook.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import display
from scipy import ndimage as ndi
from scipy import stats
from skimage import measure
from skimage.draw import polygon2mask
from skimage.feature import peak_local_max
from skimage.segmentation import watershed


### User Configuration

Set paths, shell and trace parameters, display options, and output locations.


In [ ]:
# -------------------------------
# User configuration
# -------------------------------
ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

ROI_DIR = ROOT / "results/rois/manual_bead_well_39locations"
ROI_SUMMARY_TSV = ROOT / "results/tables/manual_roi_summary_39locations.tsv"
BEAD_WELL_TSV = ROOT / "results/tables/manual_bead_well_positions_39locations.tsv"

OUT_DIR = ROOT / "results/tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Main outputs written by this notebook.
SUMMARY_CSV = OUT_DIR / "manual_gradient_per_cyst.csv"
DISTANCE_TRACE_CSV = OUT_DIR / "manual_gradient_distance_trace_long.csv"
ORIENTATION_TRACE_CSV = OUT_DIR / "manual_gradient_orientation_trace_long.csv"
ERRORS_CSV = OUT_DIR / "manual_gradient_errors.csv"
MANUAL_REVIEW_CSV = OUT_DIR / "manual_gradient_manual_review.csv"
BEAD_ASSIGNMENT_TSV = OUT_DIR / "manual_gradient_bead_assignment_per_cyst.tsv"

FIG_DIR = ROOT / "results/figures/diagnostics"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Manuscript-facing final figure directories.
FINAL_FIG_ROOT_DIR = ROOT / "results/figures"
FIG1E_DIR = FINAL_FIG_ROOT_DIR / "Fig1e_pSMAD_distance"
FIG1F_DIR = FINAL_FIG_ROOT_DIR / "Fig1f_pSMAD_orientation"
FIG1E_DIR.mkdir(parents=True, exist_ok=True)
FIG1F_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-6
PROJECTION = "max"
PIXEL_SIZE_UM_FALLBACK = None
STRICT = True

# Multi-bead bookkeeping.
PRIMARY_BEAD_AMBIGUOUS_MARGIN_UM = 100.0

# Shell sampling.
# Both distance and orientation analyses now use the same peripheral shell rather than
# the full cyst interior.
SHELL_DEPTH_PX = 55.0
SHELL_MIN_PIXELS = 120

# Trace binning.
DISTANCE_TRACE_BIN_UM = 10.0
DISTANCE_TRACE_MIN_UM = 0.0
DISTANCE_TRACE_MAX_UM = 1600.0
ORIENTATION_TRACE_BIN_DEG = 5.0
ORIENTATION_SMOOTH_WINDOW_BINS = 5

# Summary statistics.
# "ratio_peak" below is a robust peak-like summary: mean of the top 10% full-shell pixels.
PEAKLIKE_TOP_FRACTION = 0.10
ORIENTATION_NORMALIZATION_PERCENTILE = 95.0
ORIENTATION_MOMENT_BASELINE_PERCENTILE = 20.0

# No-bead controls are shown at the far right of the distance summary plot.
DISTANCE_CONTROL_OFFSET_UM = 20.0

# Plot-display only downsampling for very dense point clouds.
DISTANCE_PLOT_MAX_POINTS = 120000
ORIENTATION_PLOT_MAX_POINTS = 120000

# Standard cyst-level summary used in the main Distance Plot 2.
PLOT2_STANDARD_METRIC_COL = "ratio_near_q10_mean"
PLOT2_STANDARD_METRIC_LABEL = "Mean normalized pSMAD/DAPI in bead-nearest 10% of shell"
PLOT2_STANDARD_SHOW_CONTROLS = False  # no-bead controls do not have comparable bead-local geometry

# Global channel background normalization (single fit across the entire dataset).
BG_PARAMS_CSV = OUT_DIR / "global_channel_background_params.csv"
BACKGROUND_HIST_N_BINS = 16384
BACKGROUND_SMOOTH_SIGMA_BINS = 2.0
BACKGROUND_FIT_DISPLAY_SIGMAS = 6.0
DAPI_BG_PLOT_XLIM = (0.0, 1000.0)
PSMAD_BG_PLOT_XLIM = (0.0, 4000.0)

# Edge-only trimming for distance analysis based on aggregated per-bin pixel count.
DISTANCE_SUPPORT_WINDOW_BINS = 3
DISTANCE_SUPPORT_MIN_PIXELS_INITIAL = 900.0
DISTANCE_SUPPORT_MIN_PIXELS = float(DISTANCE_SUPPORT_MIN_PIXELS_INITIAL)
DISTANCE_SUPPORT_EDGE_BIN_FRACTION = 0.10
DISTANCE_SUPPORT_DERIVE_FROM_NONEDGE_BINS = True
DISTANCE_SUPPORT_EDGE_ONLY = True

if not ROI_DIR.exists():
    raise RuntimeError(f"Missing ROI directory: {ROI_DIR}")
if not ROI_SUMMARY_TSV.exists():
    raise RuntimeError(f"Missing ROI summary TSV: {ROI_SUMMARY_TSV}")
if not BEAD_WELL_TSV.exists():
    raise RuntimeError(f"Missing bead table: {BEAD_WELL_TSV}")

print("ROOT:", ROOT)
print("ROI_DIR:", ROI_DIR)
print("ROI_SUMMARY_TSV:", ROI_SUMMARY_TSV)
print("BEAD_WELL_TSV:", BEAD_WELL_TSV)
print("OUT_DIR:", OUT_DIR)
print("FIG_DIR:", FIG_DIR)

# Center-cyst exclusion (small central round cyst should be omitted from analysis).
OMIT_CENTER_CYST = True
CENTER_MAX_AREA_RATIO = 0.65  # candidate area / scene median area
CENTER_MAX_DIST_PX = 140.0  # candidate centroid distance to scene centroid
TANGENT_HALF_LEN_PX = 95.0  # displayed tangent half-length in geometry overlays
GEOM_MONTAGE_NCOLS = 6

# Optimization mode suppresses notebook displays/figures but still writes core CSV outputs.
# Review mode keeps the audit trail readable by muting image-heavy development sections.
OPTIMIZATION_MODE = False
REVIEW_MODE = False
RENDER_PREANALYSIS_OUTLINE_MONTAGE = (not OPTIMIZATION_MODE) and (not REVIEW_MODE)
RENDER_PREANALYSIS_GEOMETRY_MONTAGE = not OPTIMIZATION_MODE
RENDER_BACKGROUND_FIT_HISTOGRAMS = (not OPTIMIZATION_MODE) and (not REVIEW_MODE)
RENDER_EDGE_TRIM_FIGURES = (not OPTIMIZATION_MODE) and (not REVIEW_MODE)
RENDER_QC_FIGURES = (not OPTIMIZATION_MODE) and (not REVIEW_MODE)
RENDER_SENSITIVITY_ANALYSIS_FIGURES = (not OPTIMIZATION_MODE) and (not REVIEW_MODE)
SAVE_FIGURES = not OPTIMIZATION_MODE
FIG_EXPORT_DPI = 300
FIG_EXPORT_FORMATS = ("png", "pdf", "svg")

# Merged-cyst handling in pre-analysis geometry montage.
MERGED_AXIS_AREA_RATIO = 1.30  # candidate if area >= ratio * scene median area
MERGED_AXIS_MIN_PEAK_DISTANCE_PX = 70  # min spacing between lobe seeds
MERGED_AXIS_MIN_FRACTION = 0.22  # each lobe must occupy at least this mask fraction
MERGED_AXIS_MIN_OUTER_RADIUS_FRACTION = (
    0.60  # each lobe centroid must be at least this fraction of the scene outer radius
)
MERGED_AXIS_MIN_RADIUS_RATIO = (
    0.55  # reject outer+center merges when one lobe sits much closer to array center
)

### Helper Imports

Import shared helper functions and sync the notebook runtime context to that module.


In [ ]:
# -------------------------------
# Helper imports from scripts/
# -------------------------------
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts import gradient_quantification_helpers as gqh

globals().update({name: getattr(gqh, name) for name in gqh.__all__})


def sync_helper_context() -> None:
    """Push current notebook variables into the helper module runtime context."""
    runtime_keys = [
        "ROOT",
        "PROJECTION",
        "EPS",
        "CHANNEL_BG_PARAMS",
        "DISTANCE_TRACE_BIN_UM",
        "DISTANCE_TRACE_MIN_UM",
        "DISTANCE_TRACE_MAX_UM",
        "DISTANCE_PLOT_MAX_POINTS",
        "ORIENTATION_TRACE_BIN_DEG",
        "ORIENTATION_SMOOTH_WINDOW_BINS",
        "ORIENTATION_NORMALIZATION_PERCENTILE",
        "ORIENTATION_MOMENT_BASELINE_PERCENTILE",
        "PEAKLIKE_TOP_FRACTION",
        "PIXEL_SIZE_UM_FALLBACK",
        "SHELL_DEPTH_PX",
        "SHELL_MIN_PIXELS",
        "PROBLEM_EDGE_BIN_FRACTION",
        "PROBLEM_MID_BIN_FRACTION",
        "PROBLEM_BACKGROUND_RING_PX",
        "DISTANCE_SUPPORT_WINDOW_BINS",
        "DISTANCE_SUPPORT_MIN_PIXELS",
        "DISTANCE_SUPPORT_EDGE_ONLY",
        "MERGED_AXIS_AREA_RATIO",
        "MERGED_AXIS_MIN_FRACTION",
        "MERGED_AXIS_MIN_OUTER_RADIUS_FRACTION",
        "MERGED_AXIS_MIN_PEAK_DISTANCE_PX",
        "MERGED_AXIS_MIN_RADIUS_RATIO",
        "bead_df",
        "summary_df",
        "roi_array_center_map",
        "ROI_SUMMARY_TSV",
        "FIG_DIR",
        "SAVE_FIGURES",
    ]
    gqh.set_runtime_context(**{k: globals()[k] for k in runtime_keys if k in globals()})


sync_helper_context()
print("Imported helper functions from scripts/gradient_quantification_helpers.py")

### Pre-analysis Geometry And Scene Setup

Assemble the scene-level inputs used downstream: bead tables, ROI indexing, per-cyst primary-bead assignment, center-cyst exclusion maps, array-center estimates, and pre-analysis outline montages.

### Multi-Bead Assignment Note

For scenes with more than one annotated bead, each cyst is assigned a single primary bead using the minimum distance from that bead to the cyst boundary. The notebook records the second-best boundary distance and a provisional ambiguity flag so we can review borderline cases in a dedicated follow-up notebook later. The core gradient quantification here still uses one bead per cyst.

In [ ]:
# -------------------------------
# Pre-analysis setup + center-cyst exclusion maps + all-scene geometry montage
# -------------------------------
bead_long_df = (
    pd.read_csv(BEAD_WELL_TSV, sep="	")
    .sort_values(["scene_index", "bead_well_index"])
    .reset_index(drop=True)
)

required_long_cols = {
    "scene_index",
    "scene",
    "brightfield_image_path",
    "dapi_image_path",
    "psmad_image_path",
    "bead_image_path",
    "cyst_labels_path",
    "n_cysts_detected",
    "annotation_status",
    "bead_well_index",
    "bead_well_x_px",
    "bead_well_y_px",
    "pixel_size_um",
}
missing_long_cols = required_long_cols.difference(bead_long_df.columns)
if missing_long_cols:
    raise RuntimeError(
        f"Missing required columns in bead table: {sorted(missing_long_cols)}"
    )


def scene_bead_xys_from_row(sr) -> list[list[float]]:
    raw = sr.get("bead_xys_px_json", None)
    if isinstance(raw, str) and raw.strip():
        try:
            coords = json.loads(raw)
        except Exception:
            coords = None
        if isinstance(coords, list):
            out = []
            for xy in coords:
                if isinstance(xy, (list, tuple)) and len(xy) >= 2:
                    x = float(xy[0])
                    y = float(xy[1])
                    if np.isfinite(x) and np.isfinite(y):
                        out.append([x, y])
            if len(out):
                return out

    x = sr.get("bead_well_x_px", np.nan)
    y = sr.get("bead_well_y_px", np.nan)
    if np.isfinite(x) and np.isfinite(y):
        return [[float(x), float(y)]]
    return []


def scatter_scene_beads(ax, sr, color="magenta", s=50, marker="x", linewidths=1.2):
    bead_xys = scene_bead_xys_from_row(sr)
    if len(bead_xys):
        bead_arr = np.asarray(bead_xys, dtype=np.float64)
        ax.scatter(bead_arr[:, 0], bead_arr[:, 1], c=color, s=s, marker=marker, linewidths=linewidths)
    return bead_xys


def summarize_scene_bead_annotations(bead_long_table: pd.DataFrame):
    scene_rows = []
    scene_bead_records_map_local = {}

    for scene_index, g in bead_long_table.groupby("scene_index", sort=True):
        g = g.sort_values("bead_well_index").reset_index(drop=True)
        statuses = g["annotation_status"].astype(str).dropna().unique().tolist()
        if len(statuses) != 1:
            raise RuntimeError(
                f"Scene {int(scene_index):02d} has inconsistent annotation_status values: {statuses}"
            )
        status = str(statuses[0])

        bead_records = []
        if status == "annotated":
            for _, rr in g.iterrows():
                x = rr.get("bead_well_x_px", np.nan)
                y = rr.get("bead_well_y_px", np.nan)
                if np.isfinite(x) and np.isfinite(y):
                    bead_records.append(
                        {
                            "bead_well_index": int(rr.get("bead_well_index", len(bead_records) + 1)),
                            "x_px": float(x),
                            "y_px": float(y),
                        }
                    )
            if len(bead_records) == 0:
                raise RuntimeError(
                    f"Scene {int(scene_index):02d} is annotated but has no finite bead coordinates."
                )
        elif status == "no_bead":
            bead_records = []
        else:
            raise RuntimeError(
                f"Unsupported annotation_status for scene {int(scene_index):02d}: {status}"
            )

        row = g.iloc[0].to_dict()
        row["annotation_status"] = status
        row["n_beads_annotated"] = int(len(bead_records))
        row["bead_xys_px_json"] = json.dumps(
            [[float(br["x_px"]), float(br["y_px"])] for br in bead_records]
        )
        if len(bead_records) == 1:
            row["bead_well_x_px"] = float(bead_records[0]["x_px"])
            row["bead_well_y_px"] = float(bead_records[0]["y_px"])
        else:
            row["bead_well_x_px"] = np.nan
            row["bead_well_y_px"] = np.nan

        scene_rows.append(row)
        scene_bead_records_map_local[int(scene_index)] = bead_records

    scene_df = (
        pd.DataFrame(scene_rows)
        .sort_values("scene_index")
        .reset_index(drop=True)
    )
    return scene_df, scene_bead_records_map_local


bead_df, scene_bead_records_map = summarize_scene_bead_annotations(bead_long_df)
coord_ok = bead_df["annotation_status"].astype(str).eq("annotated") & (
    bead_df["n_beads_annotated"].astype(int) > 0
)

roi_files = list_roi_files_from_summary(ROI_SUMMARY_TSV, ROOT)
if len(roi_files) == 0:
    raise RuntimeError(f"No ROI files found from ROI summary TSV: {ROI_SUMMARY_TSV}")

roi_index_df = check_unique_cyst_ids_or_fail(roi_files)
expected_roi_count = int(bead_df.loc[coord_ok, "n_cysts_detected"].sum())
if len(roi_files) != expected_roi_count:
    raise RuntimeError(
        f"ROI file count mismatch: found={len(roi_files)} expected_from_bead_scenes={expected_roi_count}. "
        "Rebuild ROI files before quantification."
    )

scene_label_shape_map = {}
for _, sr in bead_df.iterrows():
    labels_path = resolve_path(str(sr["cyst_labels_path"]), ROOT)
    scene_label_shape_map[int(sr["scene_index"])] = tifffile.imread(labels_path).shape


def assign_primary_bead_to_roi_record(record: dict) -> dict:
    scene_index = int(record.get("scene_index", -1))
    cyst_id = str(record.get("cyst_id", ""))
    bead_records = scene_bead_records_map.get(scene_index, [])
    if len(bead_records) == 0:
        raise RuntimeError(
            f"ROI {cyst_id or '<unknown>'} belongs to scene {scene_index} with no annotated beads."
        )

    mask_shape = scene_label_shape_map[int(scene_index)]
    mask = polygon_xy_to_mask(mask_shape, record["polygon_xy_px"])
    boundary_xy = extract_boundary_xy(mask)
    bead_xy = np.asarray([[br["x_px"], br["y_px"]] for br in bead_records], dtype=np.float64)
    boundary_dist_px = np.linalg.norm(boundary_xy[:, None, :] - bead_xy[None, :, :], axis=2).min(axis=0)
    order = np.argsort(boundary_dist_px)

    primary_ord = int(order[0])
    primary = bead_records[primary_ord]
    primary_dist_px = float(boundary_dist_px[primary_ord])

    secondary_dist_px = np.nan
    secondary_idx = np.nan
    if len(order) > 1:
        secondary_ord = int(order[1])
        secondary = bead_records[secondary_ord]
        secondary_dist_px = float(boundary_dist_px[secondary_ord])
        secondary_idx = int(secondary["bead_well_index"])

    pixel_um = float(record.get("pixel_size_um", PIXEL_SIZE_UM_FALLBACK))
    primary_dist_um = float(primary_dist_px * pixel_um)
    secondary_dist_um = float(secondary_dist_px * pixel_um) if np.isfinite(secondary_dist_px) else np.nan
    assignment_margin_px = float(secondary_dist_px - primary_dist_px) if np.isfinite(secondary_dist_px) else np.nan
    assignment_margin_um = float(assignment_margin_px * pixel_um) if np.isfinite(assignment_margin_px) else np.nan
    ambiguous = bool(
        len(order) > 1
        and np.isfinite(assignment_margin_um)
        and assignment_margin_um <= PRIMARY_BEAD_AMBIGUOUS_MARGIN_UM
    )

    return {
        "scene_index": scene_index,
        "scene": record.get("scene", ""),
        "cyst_id": cyst_id,
        "n_candidate_beads": int(len(bead_records)),
        "primary_bead_index": int(primary["bead_well_index"]),
        "primary_bead_x_px": float(primary["x_px"]),
        "primary_bead_y_px": float(primary["y_px"]),
        "primary_bead_distance_px": float(primary_dist_px),
        "primary_bead_distance_um": float(primary_dist_um),
        "secondary_bead_index": secondary_idx,
        "secondary_bead_distance_px": secondary_dist_px,
        "secondary_bead_distance_um": secondary_dist_um,
        "assignment_margin_px": assignment_margin_px,
        "assignment_margin_um": assignment_margin_um,
        "bead_assignment_ambiguous": ambiguous,
        "bead_xys_px_json": json.dumps([[float(br["x_px"]), float(br["y_px"])] for br in bead_records]),
    }


roi_assignment_rows = []
roi_assigned_record_map = {}
for rf in roi_files:
    rec = json.loads(rf.read_text(encoding="utf-8"))
    assign = assign_primary_bead_to_roi_record(rec)
    rec_assigned = dict(rec)
    rec_assigned["bead_xy_px"] = [assign["primary_bead_x_px"], assign["primary_bead_y_px"]]
    rec_assigned["bead_well_x_px"] = float(assign["primary_bead_x_px"])
    rec_assigned["bead_well_y_px"] = float(assign["primary_bead_y_px"])
    rec_assigned["bead_well_index"] = int(assign["primary_bead_index"])
    rec_assigned["n_beads_annotated"] = int(assign["n_candidate_beads"])
    rec_assigned["bead_assignment_ambiguous"] = bool(assign["bead_assignment_ambiguous"])
    rec_assigned["bead_assignment_margin_um"] = assign["assignment_margin_um"]
    rec_assigned["secondary_bead_distance_um"] = assign["secondary_bead_distance_um"]
    rec_assigned["primary_bead_distance_um"] = assign["primary_bead_distance_um"]
    rec_assigned["primary_bead_index"] = int(assign["primary_bead_index"])
    rec_assigned["secondary_bead_index"] = assign["secondary_bead_index"]
    rec_assigned["bead_xys_px"] = json.loads(assign["bead_xys_px_json"])
    roi_assigned_record_map[str(rf)] = rec_assigned
    roi_assignment_rows.append({**assign, "roi_file": str(rf)})

roi_assignment_df = (
    pd.DataFrame(roi_assignment_rows)
    .sort_values(["scene_index", "cyst_id"])
    .reset_index(drop=True)
)
roi_assignment_df.to_csv(BEAD_ASSIGNMENT_TSV, sep="	", index=False)
roi_assignment_map = roi_assignment_df.set_index("cyst_id").to_dict(orient="index")

# Center-cyst exclusion map for bead-present ROI records (scene_index -> cyst_id).
roi_feat_df = build_roi_feature_table(roi_files)
roi_center_exclusion_df, roi_center_exclusion_map = build_scene_center_exclusion_map(
    roi_feat_df,
    id_col="cyst_id",
    area_ratio_max=CENTER_MAX_AREA_RATIO,
    dist_max_px=CENTER_MAX_DIST_PX,
)

roi_array_center_df, roi_array_center_map = build_scene_array_center_map(
    roi_feat_df,
    id_col="cyst_id",
    center_exclusion_map=roi_center_exclusion_map,
)

# Center-cyst exclusion map for full scene labels (scene_index -> label_id), used for controls and montage labels.
label_feat_rows = []
for _, sr in bead_df.iterrows():
    label_feat_rows.append(build_label_feature_table(sr, ROOT))
label_feat_df = pd.concat(label_feat_rows, ignore_index=True)
label_center_exclusion_df, label_center_exclusion_map = (
    build_scene_center_exclusion_map(
        label_feat_df,
        id_col="label_id",
        area_ratio_max=CENTER_MAX_AREA_RATIO,
        dist_max_px=CENTER_MAX_DIST_PX,
    )
)


# Force center-cyst exclusion for specific scenes.
FORCE_CENTER_CYST_SCENES = [0, 6, 7, 13, 15, 16, 18, 24]


def _force_center_exclusion_from_features(df, id_col, scenes):
    out = {}
    for sid in scenes:
        g = df.loc[df["scene_index"] == int(sid)].copy()
        if len(g) == 0:
            continue
        g = g[
            np.isfinite(g["centroid_x_px"].astype(float))
            & np.isfinite(g["centroid_y_px"].astype(float))
        ]
        if len(g) == 0:
            continue
        cx = g["centroid_x_px"].astype(float).to_numpy()
        cy = g["centroid_y_px"].astype(float).to_numpy()
        mx = float(np.nanmean(cx))
        my = float(np.nanmean(cy))
        dist = np.sqrt((cx - mx) ** 2 + (cy - my) ** 2)
        pick = g.iloc[int(np.nanargmin(dist))][id_col]
        out[int(sid)] = pick
    return out


forced_roi_center_map = _force_center_exclusion_from_features(
    roi_feat_df, id_col="cyst_id", scenes=FORCE_CENTER_CYST_SCENES
)
forced_label_center_map = _force_center_exclusion_from_features(
    label_feat_df, id_col="label_id", scenes=FORCE_CENTER_CYST_SCENES
)
if len(forced_roi_center_map):
    roi_center_exclusion_map.update(forced_roi_center_map)
if len(forced_label_center_map):
    label_center_exclusion_map.update(forced_label_center_map)

roi_array_center_df, roi_array_center_map = build_scene_array_center_map(
    roi_feat_df,
    id_col="cyst_id",
    center_exclusion_map=roi_center_exclusion_map,
)


if OMIT_CENTER_CYST:
    print("Center-cyst exclusion enabled.")
    print("ROI scenes with excluded center cyst:", len(roi_center_exclusion_map))
    print("All scenes with excluded center label:", len(label_center_exclusion_map))
else:
    print("Center-cyst exclusion disabled (OMIT_CENTER_CYST=False).")
    roi_center_exclusion_map = {}
    label_center_exclusion_map = {}

print("Bead-present scenes:", int(coord_ok.sum()))
print("No-bead scenes:", int((~coord_ok).sum()))
print("Bead-present ROI files:", len(roi_files))
print("Primary-bead assignments saved:", BEAD_ASSIGNMENT_TSV)
print("Ambiguous primary-bead assignments:", int(roi_assignment_df["bead_assignment_ambiguous"].sum()))

print()
if OPTIMIZATION_MODE:
    print("Skipping pre-analysis dataframe displays (OPTIMIZATION_MODE=True).")
else:
    print("Scene-level bead annotation summary (first 12 rows):")
    display(
        bead_df[[
            "scene_index",
            "scene",
            "annotation_status",
            "n_cysts_detected",
            "n_beads_annotated",
            "bead_xys_px_json",
        ]].head(12)
    )

    print("Primary-bead assignment summary (first 12 rows):")
    display(roi_assignment_df.head(12))

    ambiguous_df = roi_assignment_df.loc[
        roi_assignment_df["bead_assignment_ambiguous"].astype(bool)
    ].copy()
    print("Ambiguous primary-bead assignments (first 12 rows):")
    if len(ambiguous_df):
        display(ambiguous_df.head(12))
    else:
        print("None flagged with the provisional margin threshold.")

    print("ROI center-exclusion candidates (first 12 rows):")
    display(roi_center_exclusion_df.head(12))

    print("ROI array-center coordinates (first 12 rows):")
    display(roi_array_center_df.head(12))

    print("Label center-exclusion candidates (first 12 rows):")
    display(label_center_exclusion_df.head(12))

print()
if RENDER_PREANALYSIS_OUTLINE_MONTAGE:
    print("Outline montage first: BF + cyst labels + bead marker")
    plot_all_scene_outline_before_analysis(
        bead_df=bead_df,
        project_root=ROOT,
        ncols=GEOM_MONTAGE_NCOLS,
    )
else:
    print(
        "Skipping all-scene outline montage (RENDER_PREANALYSIS_OUTLINE_MONTAGE=False)."
    )

print()
if RENDER_PREANALYSIS_GEOMETRY_MONTAGE:
    print(
        "Geometry montage second: anterior=toward array center, posterior=away from array center"
    )
    print("Center cysts are shown without axis/AP points.")
    FORCE_SINGLE_AXIS_SCENES = {15, 17, 19, 23}
    FORCE_TWO_AXIS_SCENES = {27}

    def _rightmost_label_id(df, scene_index, exclude_label=None):
        g = df.loc[df["scene_index"] == int(scene_index)].copy()
        if exclude_label is not None:
            g = g.loc[g["label_id"] != int(exclude_label)]
        if len(g) == 0:
            return None
        g = g[np.isfinite(g["centroid_x_px"].astype(float))]
        if len(g) == 0:
            return None
        pick_idx = g["centroid_x_px"].astype(float).idxmax()
        return int(g.loc[pick_idx, "label_id"])

    force_two_axis_label_map = {}
    for sid in sorted(FORCE_TWO_AXIS_SCENES):
        omit_label = label_center_exclusion_map.get(int(sid), None)
        label_id = _rightmost_label_id(label_feat_df, sid, exclude_label=omit_label)
        if label_id is not None:
            force_two_axis_label_map[int(sid)] = int(label_id)

    plot_all_scenes_geometry_before_analysis(
        bead_df=bead_df,
        center_label_map=label_center_exclusion_map,
        project_root=ROOT,
        ncols=GEOM_MONTAGE_NCOLS,
        tangent_half_len_px=TANGENT_HALF_LEN_PX,
        force_single_axis_scenes=FORCE_SINGLE_AXIS_SCENES,
        force_two_axis_label_map=force_two_axis_label_map,
    )
else:
    print(
        "Skipping all-scene geometry montage (RENDER_PREANALYSIS_GEOMETRY_MONTAGE=False)."
    )

sync_helper_context()

## Quantification


### Global Channel Background Normalization

Fit one dataset-wide background model per channel. These parameters are used throughout the analysis.


In [ ]:
# -------------------------------
# Global channel background normalization
# -------------------------------
channel_bg_rows = []
channel_hist_payloads = {}
CHANNEL_BG_PARAMS = {}

for channel_name, path_col in [
    ("dapi", "dapi_image_path"),
    ("psmad", "psmad_image_path"),
]:
    image_paths = unique_scene_image_paths(bead_df, path_col, ROOT)
    fit = estimate_background_half_gaussian(
        pooled_histogram_all_pixels(
            image_paths=image_paths,
            n_bins=BACKGROUND_HIST_N_BINS,
            projection=PROJECTION,
        ),
        smooth_sigma_bins=BACKGROUND_SMOOTH_SIGMA_BINS,
    )
    CHANNEL_BG_PARAMS[channel_name] = fit
    channel_hist_payloads[channel_name] = fit
    channel_bg_rows.append(
        {
            "channel": channel_name,
            "mu_bg_raw": fit["mu_bg_raw"],
            "sigma_bg_raw": fit["sigma_bg_raw"],
            "mode_idx": fit["mode_idx"],
            "n_images": fit["n_images"],
            "n_pixels": fit["n_pixels"],
            "global_min": fit["global_min"],
            "global_max": fit["global_max"],
            "fit_method": fit["fit_method"],
        }
    )

bg_params_df = pd.DataFrame(channel_bg_rows)
bg_params_df.to_csv(BG_PARAMS_CSV, index=False)
print(bg_params_df.to_string(index=False))

if OPTIMIZATION_MODE or not RENDER_BACKGROUND_FIT_HISTOGRAMS:
    reason = "OPTIMIZATION_MODE=True" if OPTIMIZATION_MODE else "RENDER_BACKGROUND_FIT_HISTOGRAMS=False"
    print(f"Skipping background-fit figures ({reason}).")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
    for ax, channel_name in zip(axes, ["dapi", "psmad"]):
        fit = channel_hist_payloads[channel_name]
        x = np.asarray(fit["centers"], dtype=float)
        counts = np.asarray(fit["counts"], dtype=float)
        smooth = np.asarray(fit["smooth_counts"], dtype=float)
        fit_counts = np.asarray(fit["fit_counts"], dtype=float)
        mu = float(fit["mu_bg_raw"])

        ax.plot(x, counts, color="0.75", lw=1.0, label="pooled histogram")
        ax.plot(x, smooth, color="k", lw=1.5, label="smoothed histogram")
        ax.axvline(mu, color="tab:red", ls="--", lw=1.5, label="mu_bg")
        ax.plot(x, fit_counts, color="tab:blue", lw=2.0, label="left-half Gaussian")
        ax.set_title(f"{channel_name.upper()} background fit")
        ax.set_xlabel("Raw intensity")
        ax.set_ylabel("Pixel count")
        if channel_name == "dapi":
            ax.set_xlim(*DAPI_BG_PLOT_XLIM)
        else:
            ax.set_xlim(*PSMAD_BG_PLOT_XLIM)
        ax.legend(frameon=False, fontsize=8)

    plt.show()

sync_helper_context()

### Bead-Present Quantification

Quantify bead-present cysts from ROI JSONs and build the per-cyst distance and orientation trace tables.


In [ ]:
# -------------------------------
# Quantify bead-present cysts
# -------------------------------
print("ROI files:", len(roi_files))
print("Unique cyst_id check: PASS")


def run_bead_present_quantification(
    verbose: bool = True,
    measurement_cache_in: dict[str, dict] | None = None,
    reuse_cached_measurements: bool = False,
):
    summaries_local = []
    distance_trace_rows_local = []
    orientation_trace_rows_local = []
    errors_local = []
    skipped_center_local = []
    image_cache_local = {}
    measurement_cache_local = {} if measurement_cache_in is None else measurement_cache_in
    measured_from_images_n = 0
    reused_measurements_n = 0

    for rf in roi_files:
        try:
            rec = dict(roi_assigned_record_map[str(rf)])
            scene_index = int(rec.get("scene_index", -1))
            cyst_id = str(rec.get("cyst_id", rf.stem))

            if OMIT_CENTER_CYST:
                center_cyst = roi_center_exclusion_map.get(scene_index, None)
                if center_cyst is not None and str(center_cyst) == cyst_id:
                    skipped_center_local.append(
                        {
                            "scene_index": scene_index,
                            "cyst_id": cyst_id,
                            "roi_file": str(rf),
                            "skip_reason": "omit_small_center_cyst",
                        }
                    )
                    continue

            array_center_xy = get_scene_array_center_or_fail(
                scene_index=scene_index,
                array_center_map=roi_array_center_map,
                context="bead-present quantification",
            )

            cached_measurement = None
            if reuse_cached_measurements:
                cached_measurement = measurement_cache_local.get(cyst_id)
            if cached_measurement is None:
                measured_from_images_n += 1
            else:
                reused_measurements_n += 1

            s, dist_rows, orient_rows, measurement = quantify_bead_present_cyst_with_measurement(
                record=rec,
                roi_file=rf,
                project_root=ROOT,
                image_cache=image_cache_local,
                array_center_xy=array_center_xy,
                cached_measurement=cached_measurement,
            )
            measurement_cache_local[cyst_id] = measurement
            summaries_local.append(s)
            distance_trace_rows_local.extend(dist_rows)
            orientation_trace_rows_local.extend(orient_rows)
        except Exception as exc:
            msg = f"{type(exc).__name__}: {exc}"
            errors_local.append({"roi_file": str(rf), "error": msg})
            print(f"[ERROR] {rf.name}: {msg}")
            if STRICT:
                raise

    if verbose:
        print("Bead-present cyst summaries:", len(summaries_local))
        print("Distance trace rows:", len(distance_trace_rows_local))
        print("Orientation trace rows:", len(orientation_trace_rows_local))
        print("Skipped center cysts:", len(skipped_center_local))
        print("Measured from images:", measured_from_images_n)
        if reuse_cached_measurements:
            print("Reused cached shell measurements:", reused_measurements_n)
        if len(skipped_center_local):
            if OPTIMIZATION_MODE:
                print("Skipping skipped-center preview table (OPTIMIZATION_MODE=True).")
            else:
                display(
                    pd.DataFrame(skipped_center_local)
                    .sort_values(["scene_index", "cyst_id"])
                    .head(20)
                )

    return (
        summaries_local,
        distance_trace_rows_local,
        orientation_trace_rows_local,
        errors_local,
        skipped_center_local,
        measurement_cache_local,
    )


(
    summaries,
    distance_trace_rows,
    orientation_trace_rows,
    errors,
    skipped_center,
    bead_measurement_cache,
) = run_bead_present_quantification(verbose=True)

### No-Bead Control Quantification

Quantify no-bead scenes from label masks for control reference.


In [ ]:
# -------------------------------
# Quantify no-bead control scenes
# -------------------------------
control_scene_df = bead_df.loc[~coord_ok].copy().sort_values("scene_index")
control_image_cache = {}

control_summaries = []
for _, scene_row in control_scene_df.iterrows():
    excluded_label_id = None
    if OMIT_CENTER_CYST:
        excluded_label_id = label_center_exclusion_map.get(
            int(scene_row["scene_index"]), None
        )

    control_summaries.extend(
        quantify_no_bead_control_scene(
            scene_row=scene_row,
            project_root=ROOT,
            image_cache=control_image_cache,
            excluded_label_id=excluded_label_id,
        )
    )

expected_controls_raw = (
    int(control_scene_df["n_cysts_detected"].sum()) if len(control_scene_df) else 0
)
excluded_control_count = 0
if OMIT_CENTER_CYST and len(control_scene_df):
    excluded_control_count = sum(
        1
        for s in control_scene_df["scene_index"].astype(int).tolist()
        if s in label_center_exclusion_map
    )
expected_controls = int(expected_controls_raw - excluded_control_count)
if len(control_summaries) != expected_controls:
    raise RuntimeError(
        f"No-bead control count mismatch: computed={len(control_summaries)} expected={expected_controls} "
        f"(raw={expected_controls_raw}, excluded_center={excluded_control_count})"
    )

print("No-bead scenes:", len(control_scene_df))
print("No-bead control cysts:", len(control_summaries))
print("Excluded center cysts in no-bead scenes:", excluded_control_count)
if len(control_scene_df):
    if OPTIMIZATION_MODE:
        print("Skipping no-bead scene preview table (OPTIMIZATION_MODE=True).")
    else:
        display(
            control_scene_df[
                [
                    "scene_index",
                    "scene",
                    "n_cysts_detected",
                    "annotation_status",
                    "n_beads_annotated",
                ]
            ]
        )

### Build Final Tables

Freeze the calibrated edge-trim threshold, apply manual review decisions, and write the final per-cyst and trace tables to disk.


In [ ]:
# -------------------------------
# Derive strict edge-trim floor + build final tables
# -------------------------------
distance_trace_df = pd.DataFrame(distance_trace_rows)
strict_support_floor = np.nan
strict_support_floor_df = pd.DataFrame()

if DISTANCE_SUPPORT_DERIVE_FROM_NONEDGE_BINS:
    strict_support_floor, strict_support_floor_df = compute_non_edge_support_floor(
        distance_trace_df,
        edge_bin_fraction=DISTANCE_SUPPORT_EDGE_BIN_FRACTION,
    )
    if not np.isfinite(strict_support_floor):
        raise RuntimeError(
            "Could not derive strict non-edge support floor from distance traces."
        )

    print(
        "Derived strict no-interior-loss count floor from occupied non-edge bins in provisional traces:",
        f"{strict_support_floor:.1f}",
    )
    print(
        f"Initial/provisional distance-support min count: {DISTANCE_SUPPORT_MIN_PIXELS_INITIAL:.1f}"
    )

    DISTANCE_SUPPORT_MIN_PIXELS = float(strict_support_floor)
    sync_helper_context()
    print()
    print(
        "Re-summarizing cached bead-present measurements once with frozen strict no-interior-loss count floor:",
        f"{DISTANCE_SUPPORT_MIN_PIXELS:.1f}",
    )
    (
        summaries,
        distance_trace_rows,
        orientation_trace_rows,
        errors,
        skipped_center,
        bead_measurement_cache,
    ) = run_bead_present_quantification(
        verbose=True,
        measurement_cache_in=bead_measurement_cache,
        reuse_cached_measurements=True,
    )

summary_df = pd.DataFrame(summaries + control_summaries)
distance_trace_df = pd.DataFrame(distance_trace_rows)
orientation_trace_df = pd.DataFrame(orientation_trace_rows)
errors_df = pd.DataFrame(errors)

if len(summary_df) == 0:
    raise RuntimeError("No summary rows produced.")

summary_df = summary_df.merge(
    roi_assignment_df[[
        "cyst_id",
        "n_candidate_beads",
        "primary_bead_index",
        "primary_bead_distance_px",
        "primary_bead_distance_um",
        "secondary_bead_index",
        "secondary_bead_distance_px",
        "secondary_bead_distance_um",
        "assignment_margin_px",
        "assignment_margin_um",
        "bead_assignment_ambiguous",
    ]],
    on="cyst_id",
    how="left",
)
summary_df = summary_df.sort_values(["scene_index", "cyst_id"]).reset_index(drop=True)

manual_review_df = pd.DataFrame(
    columns=[
        "cyst_id",
        "review_action",
        "include_in_downstream_analysis",
        "review_reason",
    ]
)
manual_review_df.to_csv(MANUAL_REVIEW_CSV, index=False)


summary_df = summary_df.merge(
    manual_review_df,
    on="cyst_id",
    how="left",
)
summary_df["review_action"] = summary_df["review_action"].astype("string").fillna("keep")
summary_df["include_in_downstream_analysis"] = (
    summary_df["include_in_downstream_analysis"].astype("boolean").fillna(True).astype(bool)
)
summary_df["review_reason"] = summary_df["review_reason"].astype("string").fillna("")
summary_df_before_manual_exclusion = summary_df.copy()

excluded_cyst_ids = (
    manual_review_df.loc[
        ~manual_review_df["include_in_downstream_analysis"].astype(bool), "cyst_id"
    ]
    .astype(str)
    .tolist()
)
distance_trace_df = distance_trace_df.loc[
    ~distance_trace_df["cyst_id"].astype(str).isin(excluded_cyst_ids)
].copy()
orientation_trace_df = orientation_trace_df.loc[
    ~orientation_trace_df["cyst_id"].astype(str).isin(excluded_cyst_ids)
].copy()
summary_df = summary_df.loc[
    summary_df["include_in_downstream_analysis"].astype(bool)
].copy()
summary_df = summary_df.sort_values(["scene_index", "cyst_id"]).reset_index(drop=True)

measured_d = summary_df.loc[summary_df["bead_present"].astype(bool), "distance_d_um"]
measured_d = measured_d[np.isfinite(measured_d)]
if len(measured_d) == 0:
    raise RuntimeError("No finite measured distance_d_um values found.")

max_d = float(np.nanmax(measured_d))
control_d_plot = float(max_d + DISTANCE_CONTROL_OFFSET_UM)

summary_df["distance_plot_um"] = np.where(
    summary_df["bead_present"].astype(bool),
    summary_df["distance_d_um"],
    control_d_plot,
)
summary_df["distance_plot_note"] = np.where(
    summary_df["bead_present"].astype(bool),
    "measured_d",
    "no_bead_control_set_to_max_plus_offset",
)

summary_df.to_csv(SUMMARY_CSV, index=False)
distance_trace_df.to_csv(DISTANCE_TRACE_CSV, index=False)
orientation_trace_df.to_csv(ORIENTATION_TRACE_CSV, index=False)
if len(errors_df):
    errors_df.to_csv(ERRORS_CSV, index=False)

print("Manual review decisions:")
display(manual_review_df)
print()
print("---")
print("Excluded cyst IDs:", excluded_cyst_ids)
print("Total cyst rows after manual review:", len(summary_df))
print("  bead-present:", int(summary_df["bead_present"].sum()))
print("  no-bead controls:", int((~summary_df["bead_present"].astype(bool)).sum()))
print("Distance trace rows:", len(distance_trace_df))
print("Orientation trace rows:", len(orientation_trace_df))
print("Max measured d (um):", f"{max_d:.3f}")
print("Control plot d (um):", f"{control_d_plot:.3f}")
print("Active distance-support min count:", f"{DISTANCE_SUPPORT_MIN_PIXELS:.1f}")
if len(strict_support_floor_df):
    print("Strict non-edge floor rows:", len(strict_support_floor_df))
print("Summary CSV:", SUMMARY_CSV)
print("Distance trace CSV:", DISTANCE_TRACE_CSV)
print("Orientation trace CSV:", ORIENTATION_TRACE_CSV)
print("Manual review CSV:", MANUAL_REVIEW_CSV)
if len(errors_df):
    print("Errors CSV:", ERRORS_CSV)

sync_helper_context()


## Binning Geometry

These sections make the distance and orientation binning explicit before the later quality-control checks and final plots.


Shared example IDs and helper context for the geometry, quality-control, and sensitivity-analysis sections are set in the next two code cells.


In [ ]:
# -------------------------------
# Diagnostics configuration (run before plotting)
# -------------------------------
bead_summary = (
    summary_df[summary_df["bead_present"].astype(bool)].copy().reset_index(drop=True)
)
if len(bead_summary) == 0:
    raise RuntimeError("No bead-present cysts available for diagnostics.")

# Default single debug cyst: median-distance bead-present cyst.
_tmp_sorted = bead_summary.sort_values("distance_d_um").reset_index(drop=True)
DEBUG_CYST_ID = str(_tmp_sorted.iloc[len(_tmp_sorted) // 2]["cyst_id"])

# Optional manual IDs. Leave empty for auto-selection.
DEBUG_CYST_IDS_MANUAL = []

# Auto-selection counts for early multi-example debugging.
N_DEBUG_DISTANCE_EXAMPLES = 5
N_DEBUG_MISMATCH_EXAMPLES = 3

# Montage config.
DEBUG_CYST_IDS_MONTAGE = []
DEBUG_MONTAGE_N = 12

print("DEBUG_CYST_ID:", DEBUG_CYST_ID)
print(
    "Set DEBUG_CYST_ID or DEBUG_CYST_IDS_MANUAL and rerun diagnostics cells if needed."
)

# Problem-pixel diagnostics: far, weak-response cysts that should be closest to bead-negative behavior.
DEBUG_PROBLEM_CYST_IDS_MANUAL = []
N_PROBLEM_PIXEL_EXAMPLES = 4
N_SUPPORT_FLOOR_EXAMPLES = 5
DEBUG_DISTANCE_BIN_CYST_IDS_MANUAL = []
N_DISTANCE_BIN_GEOMETRY_EXAMPLES = 4
PROBLEM_EDGE_BIN_FRACTION = DISTANCE_SUPPORT_EDGE_BIN_FRACTION
PROBLEM_MID_BIN_FRACTION = 0.20
PROBLEM_BACKGROUND_RING_PX = 25

# Distance Plot 1 outlier screen.
N_DISTANCE_TRACE_OUTLIERS = 6
TRACE_OUTLIER_MIN_N_CYSTS_PER_BIN = 5
TRACE_OUTLIER_SD_FLOOR_QUANTILE = 20.0

# Later outlier screens (kept here for future passes).
BEAD_DISTANT_QUANTILE = 0.75
N_DISTANT_OUTLIERS = 6
N_CONTROL_OUTLIERS = 6


In [ ]:
# -------------------------------
# Diagnostics helpers imported from scripts/
# -------------------------------
sync_helper_context()
print("Diagnostics helpers are loaded from scripts/gradient_quantification_helpers.py")

### Distance Bin Geometry

These panels show the shell-only geometry behind the distance analysis. Each shell pixel is assigned to one occupied `10 um` local-distance bin, and the same categorical bins are shown in the shell image and the count histogram.


In [ ]:
# -------------------------------
# Distance-bin geometry snapshots: shell-only tiling of exact 10 um bins
# -------------------------------
if OPTIMIZATION_MODE:
    print("Skipping distance-bin geometry snapshots (OPTIMIZATION_MODE=True).")
else:
    if len(DEBUG_DISTANCE_BIN_CYST_IDS_MANUAL) > 0:
        dist_geom_ids = [str(c) for c in DEBUG_DISTANCE_BIN_CYST_IDS_MANUAL]
    else:
        bead_ok = summary_df[summary_df["bead_present"].astype(bool)].copy()
        by_d = bead_ok.sort_values("distance_d_um").reset_index(drop=True)
        idx = np.unique(
            np.linspace(
                0,
                max(len(by_d) - 1, 0),
                min(N_DISTANCE_BIN_GEOMETRY_EXAMPLES, len(by_d)),
                dtype=int,
            )
        )
        dist_geom_ids = by_d.iloc[idx]["cyst_id"].astype(str).tolist() if len(by_d) else []

    print("Distance-bin geometry cyst IDs:", dist_geom_ids)
    print("These figures color exact shell pixels by discrete occupied 10 um local-distance bins.")

    for cid in dist_geom_ids:
        print()
        print("-" * 80)
        print("Distance-bin geometry snapshot for", cid)
        payload = build_debug_payload(cid)
        geometry = build_distance_bin_geometry_payload(payload)
        plot_distance_bin_geometry_payload(payload, geometry)



### Orientation Bin Geometry

These panels show the shell-only geometry behind the orientation analysis. Orientation uses the same shell pixels as distance, but a different binning rule:
- distance uses local bead distance in `10 um` bins and then edge-trims weakly supported end bins,
- orientation bins the full shell by bead-relative angle in `5 deg` bins and does not use the distance edge trim.


In [ ]:
# -------------------------------
# Orientation-bin geometry snapshots: shell-only tiling of exact 5 deg bins
# -------------------------------
if OPTIMIZATION_MODE:
    print("Skipping orientation-bin geometry snapshots (OPTIMIZATION_MODE=True).")
else:
    if 'dist_geom_ids' in globals() and len(dist_geom_ids) > 0:
        orient_geom_ids = [str(c) for c in dist_geom_ids]
    elif len(DEBUG_DISTANCE_BIN_CYST_IDS_MANUAL) > 0:
        orient_geom_ids = [str(c) for c in DEBUG_DISTANCE_BIN_CYST_IDS_MANUAL]
    else:
        bead_ok = summary_df[summary_df["bead_present"].astype(bool)].copy()
        by_d = bead_ok.sort_values("distance_d_um").reset_index(drop=True)
        idx = np.unique(
            np.linspace(
                0,
                max(len(by_d) - 1, 0),
                min(N_DISTANCE_BIN_GEOMETRY_EXAMPLES, len(by_d)),
                dtype=int,
            )
        )
        orient_geom_ids = by_d.iloc[idx]["cyst_id"].astype(str).tolist() if len(by_d) else []

    print("Orientation-bin geometry cyst IDs:", orient_geom_ids)
    print("These figures color full-shell pixels by discrete 5 deg bins relative to the bead direction.")
    print("Distance and orientation start from the same shell pixels, but orientation uses full-shell 5 deg angle bins rather than 10 um distance bins.")

    for cid in orient_geom_ids:
        print()
        print("-" * 80)
        print("Orientation-bin geometry snapshot for", cid)
        payload = build_debug_payload(cid)
        geometry = build_orientation_bin_geometry_payload(payload)
        plot_orientation_bin_geometry_payload(payload, geometry)



## Calibration

This section fixes the active edge-trim threshold used downstream.


### Edge-Trim Threshold Calibration

This section defines the active global minimum count threshold.

Definitions used here:
- occupied bins = all distance bins with at least one shell pixel
- edge bins = the low-distance and high-distance outer blocks, using the same edge-fraction rule as the active trim logic
- non-edge bins = all occupied bins that are not edge bins

The threshold is derived once from the provisional bead-present traces as the lowest `3-bin` aggregated count support among occupied non-edge bins. It is then frozen and the bead-present quantification is rerun once with that value. The threshold is not recalibrated after trimming.

Quick column guide:
- `distance_local_um`: center of the exact `10 um` local-distance bin
- `n_pixels`: raw shell-pixel count in that exact bin
- `count_support_n_pixels`: `3-bin` aggregated count support used by the trim rule
- `n_occupied_bins`: total occupied distance bins for that cyst
- `edge_n`: occupied bins per side treated as edge bins for that cyst
- `interior_rank`: order of this bin within the remaining non-edge block


In [ ]:
# -------------------------------
# Edge-trim threshold calibration: exact bins setting the strict global floor
# -------------------------------
if OPTIMIZATION_MODE:
    print("Skipping support-floor calibration diagnostics (OPTIMIZATION_MODE=True).")
else:
    if len(strict_support_floor_df) == 0:
        print("No strict support-floor rows available.")
    else:
        print(
            f"Active strict no-interior-loss count floor = {DISTANCE_SUPPORT_MIN_PIXELS:.1f}"
        )
        print(
            f"Derived once from provisional occupied non-edge bins with edge fraction = {DISTANCE_SUPPORT_EDGE_BIN_FRACTION:.2f}"
        )
        print(
            f"Initial/provisional count floor was {DISTANCE_SUPPORT_MIN_PIXELS_INITIAL:.1f}"
        )
        print()
        print("Lowest occupied non-edge bins setting this requirement:")
        print("  distance_local_um = exact 10 um bin center")
        print("  n_pixels = raw shell-pixel count in that exact bin")
        print("  count_support_n_pixels = 3-bin aggregated count support used by the trim rule")
        print("  n_occupied_bins = total occupied distance bins in that cyst")
        print("  edge_n = occupied bins per side treated as edge bins")
        print("  interior_rank = order of this bin within the non-edge block")
        display(
            strict_support_floor_df.head(10)[
                [
                    "cyst_id",
                    "scene_index",
                    "distance_local_um",
                    "n_pixels",
                    "count_support_n_pixels",
                    "n_occupied_bins",
                    "edge_n",
                    "interior_rank",
                ]
            ]
        )

        if not RENDER_EDGE_TRIM_FIGURES:
            print("Skipping edge-trim example figures (RENDER_EDGE_TRIM_FIGURES=False).")
        else:
            for _, ex in strict_support_floor_df.head(N_SUPPORT_FLOOR_EXAMPLES).iterrows():
                cid = str(ex["cyst_id"])
                print()
                print("-" * 80)
                print(
                    "Strict-floor example |",
                    cid,
                    "| scene=",
                    int(ex["scene_index"]),
                    "| distance bin=",
                    f"{float(ex['distance_local_um']):.1f} um",
                    "| n_pixels=",
                    int(ex["n_pixels"]),
                    "| agg support=",
                    f"{float(ex['count_support_n_pixels']):.1f}",
                )
                payload = build_debug_payload(cid)
                example = build_support_floor_example_payload(
                    payload,
                    distance_local_um=float(ex["distance_local_um"]),
                    support_floor=float(DISTANCE_SUPPORT_MIN_PIXELS),
                    edge_bin_fraction=DISTANCE_SUPPORT_EDGE_BIN_FRACTION,
                )
                plot_support_floor_example_payload(payload, example)


## Quality Control

These sections document scene-level imaging checks, problem pixels, per-cyst panels, and outlier review.


### Manual Review Exclusions

These cysts were manually excluded from downstream quantitative analysis. They are shown explicitly here so the exclusion step is auditable in the notebook itself.


In [ ]:
# -------------------------------
# Manual review exclusions: explicit audit trail before downstream QC/plots
# -------------------------------
if OPTIMIZATION_MODE:
    print("Skipping manual review exclusion diagnostics (OPTIMIZATION_MODE=True).")
else:
    manual_exclusion_df = (
        summary_df_before_manual_exclusion.loc[
            ~summary_df_before_manual_exclusion["include_in_downstream_analysis"].astype(bool)
        ]
        .copy()
        .sort_values(["scene_index", "cyst_id"])
        .reset_index(drop=True)
    )

    print("Manual review exclusions carried forward from Build Final Tables:")
    if len(manual_exclusion_df) == 0:
        print("  No cysts are currently excluded from downstream analysis.")
    else:
        display(
            manual_exclusion_df[
                [
                    "cyst_id",
                    "scene_index",
                    "bead_present",
                    "distance_d_um",
                    "review_action",
                    "review_reason",
                ]
            ]
        )

        excluded_plot_ids = (
            manual_exclusion_df.loc[
                manual_exclusion_df["bead_present"].astype(bool), "cyst_id"
            ]
            .astype(str)
            .tolist()
        )
        print("Excluded bead-present cyst IDs:", excluded_plot_ids)

        if not RENDER_QC_FIGURES:
            print("Skipping excluded-cyst figures (RENDER_QC_FIGURES=False).")
        else:
            print()
            print("=" * 80)
            print("Compact DAPI/pSMAD montage of manually excluded cysts")
            plot_geometry_montage(
                excluded_plot_ids,
                ncols=3,
                summary_source_df=summary_df_before_manual_exclusion,
            )

            print()
            print("=" * 80)
            print("Deep-dive panels for manually excluded cysts")
            for cid in excluded_plot_ids:
                print()
                print("-" * 80)
                exclusion_row = manual_exclusion_df.loc[
                    manual_exclusion_df["cyst_id"].astype(str) == str(cid)
                ].iloc[0]
                print(
                    f"Manual exclusion | {cid} | scene={int(exclusion_row['scene_index'])} | "
                    f"reason={str(exclusion_row['review_reason'])}"
                )
                payload = build_debug_payload(
                    cid,
                    summary_source_df=summary_df_before_manual_exclusion,
                )
                plot_debug_payload(
                    payload,
                    scene_rows_source_df=summary_df_before_manual_exclusion,
                )


### Scene-Level Imaging Variability QC

These outputs ask whether scene-to-scene imaging differences are large enough to justify per-image preprocessing. We retained global channel normalization, so this section is quality control only.


In [ ]:
# -------------------------------
# Per-image background drift diagnostics (raw scenes only)
# -------------------------------
PER_IMAGE_BG_HIST_N_BINS = min(int(BACKGROUND_HIST_N_BINS), 4096)
SCENE_BG_DIAG_CSV = OUT_DIR / "scene_channel_background_diagnostics.csv"

scene_image_df = (
    bead_df[["scene_index", "scene", "dapi_image_path", "psmad_image_path", "cyst_labels_path"]]
    .drop_duplicates("scene_index")
    .sort_values("scene_index")
    .reset_index(drop=True)
)

def histogram_from_pixel_values(values, n_bins=4096):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise RuntimeError("No finite pixel values available for histogram fit.")
    global_min = float(np.min(values))
    global_max = float(np.max(values))
    if global_max <= global_min:
        global_max = global_min + 1.0
    edges = np.linspace(global_min, global_max, int(n_bins) + 1, dtype=np.float64)
    counts, _ = np.histogram(values, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return {
        "edges": edges,
        "centers": centers,
        "counts": counts.astype(np.int64),
        "global_min": global_min,
        "global_max": global_max,
        "n_pixels": int(values.size),
        "n_images": 1,
        "n_bins": int(n_bins),
    }

scene_channel_bg_rows = []
for _, sr in scene_image_df.iterrows():
    labels = tifffile.imread(resolve_path(str(sr["cyst_labels_path"]), ROOT)).astype(np.int32)
    bg_keep_mask = labels <= 0
    frac_pixels_excluded = 1.0 - float(np.mean(bg_keep_mask))
    n_bg_pixels = int(np.sum(bg_keep_mask))

    for channel_name, path_col in [("dapi", "dapi_image_path"), ("psmad", "psmad_image_path")]:
        image_path = resolve_path(str(sr[path_col]), ROOT)
        raw = load_image_2d(image_path, projection=PROJECTION).astype(np.float32)

        fit_all = estimate_background_half_gaussian(
            pooled_histogram_all_pixels(
                image_paths=[image_path],
                n_bins=PER_IMAGE_BG_HIST_N_BINS,
                projection=PROJECTION,
            ),
            smooth_sigma_bins=BACKGROUND_SMOOTH_SIGMA_BINS,
        )
        fit_bgonly = estimate_background_half_gaussian(
            histogram_from_pixel_values(raw[bg_keep_mask], n_bins=PER_IMAGE_BG_HIST_N_BINS),
            smooth_sigma_bins=BACKGROUND_SMOOTH_SIGMA_BINS,
        )
        global_fit = CHANNEL_BG_PARAMS[channel_name]
        scene_channel_bg_rows.append(
            {
                "scene_index": int(sr["scene_index"]),
                "scene": sr["scene"],
                "channel": channel_name,
                "mu_bg_raw_scene": float(fit_all["mu_bg_raw"]),
                "sigma_bg_raw_scene": float(fit_all["sigma_bg_raw"]),
                "mu_bg_raw_scene_bgonly": float(fit_bgonly["mu_bg_raw"]),
                "sigma_bg_raw_scene_bgonly": float(fit_bgonly["sigma_bg_raw"]),
                "mu_bg_raw_global": float(global_fit["mu_bg_raw"]),
                "sigma_bg_raw_global": float(global_fit["sigma_bg_raw"]),
                "delta_mu_bg_raw": float(fit_all["mu_bg_raw"] - global_fit["mu_bg_raw"]),
                "delta_sigma_bg_raw": float(fit_all["sigma_bg_raw"] - global_fit["sigma_bg_raw"]),
                "delta_mu_bg_raw_bgonly_vs_all": float(fit_bgonly["mu_bg_raw"] - fit_all["mu_bg_raw"]),
                "delta_sigma_bg_raw_bgonly_vs_all": float(fit_bgonly["sigma_bg_raw"] - fit_all["sigma_bg_raw"]),
                "mean_raw": float(np.nanmean(raw)),
                "median_raw": float(np.nanmedian(raw)),
                "p95_raw": float(np.nanpercentile(raw, 95)),
                "n_pixels": int(raw.size),
                "n_bg_pixels": n_bg_pixels,
                "frac_pixels_excluded_from_bg": frac_pixels_excluded,
            }
        )

scene_channel_bg_df = (
    pd.DataFrame(scene_channel_bg_rows)
    .sort_values(["channel", "scene_index"])
    .reset_index(drop=True)
)
scene_channel_bg_df.to_csv(SCENE_BG_DIAG_CSV, index=False)

scene_bg_wide_df = scene_channel_bg_df.pivot(index="scene_index", columns="channel")
scene_bg_wide_df.columns = ["_".join(col) for col in scene_bg_wide_df.columns]
scene_bg_wide_df = scene_bg_wide_df.reset_index().sort_values("scene_index").reset_index(drop=True)

print("Saved per-image background diagnostics:", SCENE_BG_DIAG_CSV)
for channel_name in ["dapi", "psmad"]:
    sub = scene_channel_bg_df[scene_channel_bg_df["channel"] == channel_name].copy()
    print()
    print(
        f"{channel_name.upper()} all-pixel mu range={sub['mu_bg_raw_scene'].min():.1f} to {sub['mu_bg_raw_scene'].max():.1f} | bg-only minus all median={sub['delta_mu_bg_raw_bgonly_vs_all'].median():+.2f} | max abs shift={sub['delta_mu_bg_raw_bgonly_vs_all'].abs().max():.2f}"
    )

if OPTIMIZATION_MODE or not RENDER_QC_FIGURES:
    reason = "OPTIMIZATION_MODE=True" if OPTIMIZATION_MODE else "RENDER_QC_FIGURES=False"
    print(f"Skipping per-image drift figures ({reason}).")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    for row_idx, channel_name in enumerate(["dapi", "psmad"]):
        sub = scene_channel_bg_df[scene_channel_bg_df["channel"] == channel_name].copy()
        ax0 = axes[row_idx, 0]
        ax0.plot(sub["scene_index"], sub["mu_bg_raw_scene"], color="tab:blue", lw=1.6, marker="o", ms=4, label="all pixels")
        ax0.plot(sub["scene_index"], sub["mu_bg_raw_scene_bgonly"], color="tab:orange", lw=1.3, marker="s", ms=3.5, label="outside cyst labels only")
        ax0.axhline(float(CHANNEL_BG_PARAMS[channel_name]["mu_bg_raw"]), color="tab:red", ls="--", lw=1.4, label="global background mean")
        ax0.set_title(f"{channel_name.upper()} per-scene background mean")
        ax0.set_xlabel("Scene index")
        ax0.set_ylabel("Raw intensity (a.u.)")
        ax0.legend(frameon=False, fontsize=8, loc="best")

        ax1 = axes[row_idx, 1]
        ax1.axhline(0.0, color="0.7", lw=1.0)
        ax1.plot(sub["scene_index"], sub["delta_mu_bg_raw_bgonly_vs_all"], color="tab:purple", lw=1.6, marker="o", ms=4)
        ax1.set_title(f"{channel_name.upper()} bg-only minus all-pixel mean")
        ax1.set_xlabel("Scene index")
        ax1.set_ylabel("Delta raw intensity (a.u.)")
    plt.show()



In [ ]:
# -------------------------------
# Image check for background-mean outlier scenes
# -------------------------------
N_BG_IMAGE_EXAMPLES_PER_TAIL = 4

scene_plot_df = (
    bead_df[[
        "scene_index",
        "scene",
        "dapi_image_path",
        "psmad_image_path",
        "bead_xys_px_json",
        "n_beads_annotated",
        "annotation_status",
    ]]
    .drop_duplicates("scene_index")
    .sort_values("scene_index")
    .reset_index(drop=True)
)

bg_scene_reason_rows = []
for channel_name in ["dapi", "psmad"]:
    sub = scene_channel_bg_df[scene_channel_bg_df["channel"] == channel_name].copy()
    sub = sub.sort_values("mu_bg_raw_scene").reset_index(drop=True)
    low = sub.head(N_BG_IMAGE_EXAMPLES_PER_TAIL).copy()
    high = sub.tail(N_BG_IMAGE_EXAMPLES_PER_TAIL).sort_values("mu_bg_raw_scene", ascending=False).copy()
    for rank, (_, rr) in enumerate(low.iterrows(), start=1):
        bg_scene_reason_rows.append({
            "scene_index": int(rr["scene_index"]),
            "reason": f"lowest {channel_name.upper()} bg mean #{rank}",
            "channel": channel_name,
            "mu_bg_raw_scene": float(rr["mu_bg_raw_scene"]),
        })
    for rank, (_, rr) in enumerate(high.iterrows(), start=1):
        bg_scene_reason_rows.append({
            "scene_index": int(rr["scene_index"]),
            "reason": f"highest {channel_name.upper()} bg mean #{rank}",
            "channel": channel_name,
            "mu_bg_raw_scene": float(rr["mu_bg_raw_scene"]),
        })

bg_scene_reason_df = pd.DataFrame(bg_scene_reason_rows)
bg_scene_selected_df = (
    bg_scene_reason_df.groupby("scene_index")
    .agg(
        reason_summary=("reason", lambda s: "; ".join(sorted(set(map(str, s))))),
    )
    .reset_index()
)

dapi_scene_bg_df = scene_channel_bg_df[scene_channel_bg_df["channel"] == "dapi"][["scene_index", "mu_bg_raw_scene", "mu_bg_raw_scene_bgonly", "sigma_bg_raw_scene", "sigma_bg_raw_scene_bgonly", "p95_raw"]].rename(columns={
        "mu_bg_raw_scene": "dapi_mu_bg_raw_scene",
        "mu_bg_raw_scene_bgonly": "dapi_mu_bg_raw_scene_bgonly",
        "sigma_bg_raw_scene": "dapi_sigma_bg_raw_scene",
        "sigma_bg_raw_scene_bgonly": "dapi_sigma_bg_raw_scene_bgonly",
        "p95_raw": "dapi_p95_raw",
    })
psmad_scene_bg_df = scene_channel_bg_df[scene_channel_bg_df["channel"] == "psmad"][["scene_index", "mu_bg_raw_scene", "mu_bg_raw_scene_bgonly", "sigma_bg_raw_scene", "sigma_bg_raw_scene_bgonly", "p95_raw"]].rename(columns={
        "mu_bg_raw_scene": "psmad_mu_bg_raw_scene",
        "mu_bg_raw_scene_bgonly": "psmad_mu_bg_raw_scene_bgonly",
        "sigma_bg_raw_scene": "psmad_sigma_bg_raw_scene",
        "sigma_bg_raw_scene_bgonly": "psmad_sigma_bg_raw_scene_bgonly",
        "p95_raw": "psmad_p95_raw",
    })

bg_scene_plot_df = (
    bg_scene_selected_df
    .merge(scene_plot_df, on="scene_index", how="left")
    .merge(dapi_scene_bg_df, on="scene_index", how="left")
    .merge(psmad_scene_bg_df, on="scene_index", how="left")
    .sort_values("scene_index")
    .reset_index(drop=True)
)

bg_scene_raw_images = {}
for _, sr in bg_scene_plot_df.iterrows():
    sid = int(sr["scene_index"])
    bg_scene_raw_images[sid] = {
        "dapi": load_image_2d(resolve_path(str(sr["dapi_image_path"]), ROOT), projection=PROJECTION).astype(np.float32),
        "psmad": load_image_2d(resolve_path(str(sr["psmad_image_path"]), ROOT), projection=PROJECTION).astype(np.float32),
    }

def _shared_limits(scene_map, channel, lo=2.0, hi=99.7):
    vals = np.concatenate([scene_map[sid][channel].ravel() for sid in scene_map])
    vals = vals[np.isfinite(vals)]
    return float(np.nanpercentile(vals, lo)), float(np.nanpercentile(vals, hi))

bg_dapi_vmin, bg_dapi_vmax = _shared_limits(bg_scene_raw_images, "dapi")
bg_psmad_vmin, bg_psmad_vmax = _shared_limits(bg_scene_raw_images, "psmad")

print("Scenes shown for background-mean image check:")
display(bg_scene_plot_df[["scene_index", "reason_summary", "dapi_mu_bg_raw_scene", "dapi_mu_bg_raw_scene_bgonly", "psmad_mu_bg_raw_scene", "psmad_mu_bg_raw_scene_bgonly"]])

if OPTIMIZATION_MODE or not RENDER_QC_FIGURES:
    reason = "OPTIMIZATION_MODE=True" if OPTIMIZATION_MODE else "RENDER_QC_FIGURES=False"
    print(f"Skipping background-mean image montage ({reason}).")
else:
    nrows = len(bg_scene_plot_df)
    fig, axes = plt.subplots(nrows, 2, figsize=(10.8, 3.7 * nrows), constrained_layout=True)
    if nrows == 1:
        axes = np.asarray([axes])

    dapi_mappable = None
    psmad_mappable = None
    for ax_row, (_, sr) in zip(axes, bg_scene_plot_df.iterrows()):
        sid = int(sr["scene_index"])
        dapi_img = bg_scene_raw_images[sid]["dapi"]
        psmad_img = bg_scene_raw_images[sid]["psmad"]
        bead_xys = scene_bead_xys_from_row(sr)
        title_prefix = (
            f"S{sid:02d} | {sr['reason_summary']}\n"
            f"DAPI bg all={float(sr['dapi_mu_bg_raw_scene']):.1f}, outside={float(sr['dapi_mu_bg_raw_scene_bgonly']):.1f} | "
            f"pSMAD bg all={float(sr['psmad_mu_bg_raw_scene']):.1f}, outside={float(sr['psmad_mu_bg_raw_scene_bgonly']):.1f}"
        )

        dapi_mappable = ax_row[0].imshow(dapi_img, cmap="gray", vmin=bg_dapi_vmin, vmax=bg_dapi_vmax)
        if len(bead_xys):
            bead_arr = np.asarray(bead_xys, dtype=np.float64)
            ax_row[0].scatter(bead_arr[:, 0], bead_arr[:, 1], c="magenta", s=50, marker="x")
        ax_row[0].set_title(title_prefix + " | DAPI raw")
        ax_row[0].axis("off")

        psmad_mappable = ax_row[1].imshow(psmad_img, cmap="magma", vmin=bg_psmad_vmin, vmax=bg_psmad_vmax)
        if len(bead_xys):
            bead_arr = np.asarray(bead_xys, dtype=np.float64)
            ax_row[1].scatter(bead_arr[:, 0], bead_arr[:, 1], c="cyan", s=50, marker="x")
        ax_row[1].set_title(title_prefix + " | pSMAD raw")
        ax_row[1].axis("off")

    cbar1 = fig.colorbar(dapi_mappable, ax=axes[:, 0], fraction=0.018, pad=0.01)
    cbar1.set_label("DAPI raw intensity (a.u.)")
    cbar2 = fig.colorbar(psmad_mappable, ax=axes[:, 1], fraction=0.018, pad=0.01)
    cbar2.set_label("pSMAD raw intensity (a.u.)")
    plt.show()



In [ ]:
# -------------------------------
# Prepare scene IDs for the whole-image comparison below
# -------------------------------
SCENE_VARIABILITY_DIAG_CSV = OUT_DIR / "scene_variability_diagnostics.csv"

bead_only_scene_diag = summary_df[summary_df["bead_present"].astype(bool)].copy()
if len(bead_only_scene_diag) == 0:
    raise RuntimeError("No bead-present cysts available for scene-level diagnostics.")

x = bead_only_scene_diag["distance_d_um"].to_numpy(dtype=float)
y_peak = bead_only_scene_diag["ratio_peak"].to_numpy(dtype=float)
peak_slope_scene, peak_intercept_scene = np.polyfit(x, y_peak, deg=1)
bead_only_scene_diag["pred_ratio_peak_distance_only"] = peak_slope_scene * x + peak_intercept_scene
bead_only_scene_diag["resid_ratio_peak_distance_only"] = (
    bead_only_scene_diag["ratio_peak"] - bead_only_scene_diag["pred_ratio_peak_distance_only"]
)

bead_only_scene_diag["log_roi_area_px"] = np.log(bead_only_scene_diag["roi_area_px"].astype(float))
X = np.column_stack([
    np.ones(len(bead_only_scene_diag), dtype=float),
    bead_only_scene_diag["distance_d_um"].to_numpy(dtype=float),
    bead_only_scene_diag["log_roi_area_px"].to_numpy(dtype=float),
])
beta_peak = np.linalg.lstsq(X, y_peak, rcond=None)[0]
bead_only_scene_diag["pred_ratio_peak_distance_area"] = X @ beta_peak
bead_only_scene_diag["resid_ratio_peak_distance_area"] = (
    bead_only_scene_diag["ratio_peak"] - bead_only_scene_diag["pred_ratio_peak_distance_area"]
)

y_shell = bead_only_scene_diag["ratio_mean_shell"].to_numpy(dtype=float)
beta_shell = np.linalg.lstsq(X, y_shell, rcond=None)[0]
bead_only_scene_diag["pred_ratio_mean_shell_distance_area"] = X @ beta_shell
bead_only_scene_diag["resid_ratio_mean_shell_distance_area"] = (
    bead_only_scene_diag["ratio_mean_shell"] - bead_only_scene_diag["pred_ratio_mean_shell_distance_area"]
)

scene_resid_df = (
    bead_only_scene_diag.groupby("scene_index")
    .agg(
        n_cysts=("cyst_id", "size"),
        mean_distance_d_um=("distance_d_um", "mean"),
        mean_roi_area_px=("roi_area_px", "mean"),
        mean_ratio_peak=("ratio_peak", "mean"),
        mean_resid_peak_distance_only=("resid_ratio_peak_distance_only", "mean"),
        sd_resid_peak_distance_only=("resid_ratio_peak_distance_only", "std"),
        mean_resid_peak_distance_area=("resid_ratio_peak_distance_area", "mean"),
        mean_resid_shell_distance_area=("resid_ratio_mean_shell_distance_area", "mean"),
    )
    .reset_index()
)
scene_resid_df["sem_resid_peak_distance_only"] = (
    scene_resid_df["sd_resid_peak_distance_only"] / np.sqrt(scene_resid_df["n_cysts"].astype(float))
)
scene_resid_df["z_scene_resid_peak_distance_only"] = (
    scene_resid_df["mean_resid_peak_distance_only"] / scene_resid_df["sem_resid_peak_distance_only"]
)

control_scene_diag_df = (
    summary_df.loc[~summary_df["bead_present"].astype(bool)]
    .groupby("scene_index")
    .agg(
        n_controls=("cyst_id", "size"),
        control_mean_ratio_peak=("ratio_peak", "mean"),
        control_max_ratio_peak=("ratio_peak", "max"),
        control_mean_ratio_mean_shell=("ratio_mean_shell", "mean"),
    )
    .reset_index()
)

scene_variability_df = (
    scene_resid_df
    .merge(
        scene_bg_wide_df,
        on="scene_index",
        how="left",
    )
    .merge(control_scene_diag_df, on="scene_index", how="left")
    .sort_values("scene_index")
    .reset_index(drop=True)
)
scene_variability_df.to_csv(SCENE_VARIABILITY_DIAG_CSV, index=False)

print("Prepared scene-level image comparison below.")
print("High/low scenes are chosen behind the scenes from the current distance-response residuals; the reader-facing evidence is the image montage below.")


In [ ]:
# -------------------------------
# Image-based scene drift check
# -------------------------------
N_SCENE_DRIFT_EXAMPLES_PER_SIDE = 2

positive_scene_ids = (
    scene_variability_df.sort_values("mean_resid_peak_distance_only", ascending=False)["scene_index"]
    .head(N_SCENE_DRIFT_EXAMPLES_PER_SIDE)
    .astype(int)
    .tolist()
)
negative_scene_ids = (
    scene_variability_df.sort_values("mean_resid_peak_distance_only", ascending=True)["scene_index"]
    .head(N_SCENE_DRIFT_EXAMPLES_PER_SIDE)
    .astype(int)
    .tolist()
)
control_scene_ids = (
    control_scene_diag_df["scene_index"].astype(int).tolist() if len(control_scene_diag_df) else []
)
scene_drift_ids = []
for sid in positive_scene_ids + negative_scene_ids + control_scene_ids:
    if sid not in scene_drift_ids:
        scene_drift_ids.append(int(sid))

scene_drift_df = (
    scene_image_df.merge(scene_variability_df, on="scene_index", how="left")
    .merge(control_scene_diag_df, on="scene_index", how="left")
    .loc[lambda d: d["scene_index"].isin(scene_drift_ids)]
    .copy()
)
scene_drift_df["scene_index"] = pd.Categorical(scene_drift_df["scene_index"], categories=scene_drift_ids, ordered=True)
scene_drift_df = scene_drift_df.sort_values("scene_index").reset_index(drop=True)

scene_raw_images = {}
for _, sr in scene_drift_df.iterrows():
    sid = int(sr["scene_index"])
    scene_raw_images[sid] = {
        "dapi": load_image_2d(resolve_path(str(sr["dapi_image_path"]), ROOT), projection=PROJECTION).astype(np.float32),
        "psmad": load_image_2d(resolve_path(str(sr["psmad_image_path"]), ROOT), projection=PROJECTION).astype(np.float32),
    }

def _shared_limits(scene_map, channel, lo=2.0, hi=99.7):
    vals = np.concatenate([scene_map[sid][channel].ravel() for sid in scene_map])
    vals = vals[np.isfinite(vals)]
    return float(np.nanpercentile(vals, lo)), float(np.nanpercentile(vals, hi))

dapi_vmin, dapi_vmax = _shared_limits(scene_raw_images, "dapi")
psmad_vmin, psmad_vmax = _shared_limits(scene_raw_images, "psmad")

print("Scenes shown (high residual, low residual, then no-bead controls):", scene_drift_ids)

if OPTIMIZATION_MODE or not RENDER_QC_FIGURES:
    reason = "OPTIMIZATION_MODE=True" if OPTIMIZATION_MODE else "RENDER_QC_FIGURES=False"
    print(f"Skipping scene drift image montage ({reason}).")
else:
    nrows = len(scene_drift_df)
    fig, axes = plt.subplots(nrows, 2, figsize=(10.5, 3.6 * nrows), constrained_layout=True)
    if nrows == 1:
        axes = np.asarray([axes])

    dapi_mappable = None
    psmad_mappable = None

    for ax_row, (_, sr) in zip(axes, scene_drift_df.iterrows()):
        sid = int(sr["scene_index"])
        dapi_img = scene_raw_images[sid]["dapi"]
        psmad_img = scene_raw_images[sid]["psmad"]
        bead_xys = scene_bead_xys_from_row(sr)
        label_prefix = f"S{sid:02d}"
        if np.isfinite(sr.get("mean_resid_peak_distance_only", np.nan)):
            label_prefix += f" | mean resid={float(sr['mean_resid_peak_distance_only']):+.3f}"
        elif np.isfinite(sr.get("control_mean_ratio_peak", np.nan)):
            label_prefix += f" | control mean peak={float(sr['control_mean_ratio_peak']):.3f}"

        dapi_mappable = ax_row[0].imshow(dapi_img, cmap="gray", vmin=dapi_vmin, vmax=dapi_vmax)
        if len(bead_xys):
            bead_arr = np.asarray(bead_xys, dtype=np.float64)
            ax_row[0].scatter(bead_arr[:, 0], bead_arr[:, 1], c="magenta", s=50, marker="x")
        ax_row[0].set_title(label_prefix + " | DAPI raw")
        ax_row[0].axis("off")

        psmad_mappable = ax_row[1].imshow(psmad_img, cmap="magma", vmin=psmad_vmin, vmax=psmad_vmax)
        if len(bead_xys):
            bead_arr = np.asarray(bead_xys, dtype=np.float64)
            ax_row[1].scatter(bead_arr[:, 0], bead_arr[:, 1], c="cyan", s=50, marker="x")
        ax_row[1].set_title(label_prefix + " | pSMAD raw")
        ax_row[1].axis("off")

    cbar1 = fig.colorbar(dapi_mappable, ax=axes[:, 0], fraction=0.018, pad=0.01)
    cbar1.set_label("DAPI raw intensity (a.u.)")
    cbar2 = fig.colorbar(psmad_mappable, ax=axes[:, 1], fraction=0.018, pad=0.01)
    cbar2.set_label("pSMAD raw intensity (a.u.)")
    plt.show()


### Problem-Pixel Diagnostics

These panels show which shell-distance bins are excluded by the active edge-count rule and whether those exclusions look spatially and intensity-wise reasonable.


In [ ]:
# -------------------------------
# Problem-pixel diagnostics: far, weak-response cysts
# -------------------------------
if OPTIMIZATION_MODE:
    print("Skipping problem-pixel diagnostics (OPTIMIZATION_MODE=True).")
else:
    bead_ok = (
        summary_df[summary_df["bead_present"].astype(bool)]
        .copy()
        .reset_index(drop=True)
    )
    far_cut = float(bead_ok["distance_d_um"].quantile(0.75))
    far_df = bead_ok[bead_ok["distance_d_um"] >= far_cut].copy()
    peak_cut = float(far_df["ratio_peak"].median())
    calib_df = far_df[far_df["ratio_peak"] <= peak_cut].copy()

    if len(DEBUG_PROBLEM_CYST_IDS_MANUAL) > 0:
        problem_ids = [str(c) for c in DEBUG_PROBLEM_CYST_IDS_MANUAL]
        problem_rank_df = pd.DataFrame()
    else:
        problem_rows = []
        for cid in calib_df["cyst_id"].astype(str).tolist():
            payload = build_debug_payload(cid)
            problem = build_problem_pixel_payload(payload)
            problem_rows.append(
                {
                    "cyst_id": cid,
                    "scene_index": int(payload["row"]["scene_index"]),
                    "distance_d_um": float(payload["row"]["distance_d_um"]),
                    "ratio_peak": float(payload["row"]["ratio_peak"]),
                    "edge_uplift": float(problem["edge_uplift"]),
                    "left_minus_mid": float(problem["left_minus_mid"]),
                    "right_minus_mid": float(problem["right_minus_mid"]),
                }
            )
        problem_rank_df = pd.DataFrame(problem_rows).sort_values(
            "edge_uplift", ascending=False
        )
        problem_ids = (
            problem_rank_df.head(N_PROBLEM_PIXEL_EXAMPLES)["cyst_id"]
            .astype(str)
            .tolist()
        )

    print("Problem-pixel calibration subset:")
    print(f"  far-cut distance >= {far_cut:.1f} um")
    print(f"  weak-response cutoff ratio_peak <= {peak_cut:.3f}")
    print("Problem-pixel diagnostic cyst IDs:", problem_ids)

    if not RENDER_QC_FIGURES:
        print("Skipping problem-pixel figures (RENDER_QC_FIGURES=False).")
    else:
        for cid in problem_ids:
            print()
            print("-" * 80)
            print("Problem-pixel diagnostic for", cid)
            payload = build_debug_payload(cid)
            problem = build_problem_pixel_payload(payload)
            plot_problem_pixel_payload(payload, problem)

    if len(problem_rank_df) > 0:
        print()
        print("Top calibration cysts by edge uplift:")
        display(problem_rank_df.head(12))

### Detailed Per-Cyst Diagnostics

Inspect full per-cyst panels once the shell geometry and edge-trim logic look reasonable.


In [ ]:
# -------------------------------
# Early diagnostics: detailed per-cyst panels BEFORE traces/plots
# -------------------------------
if OPTIMIZATION_MODE:
    print("Skipping detailed per-cyst diagnostics (OPTIMIZATION_MODE=True).")
else:
    bead_ok = (
        summary_df[summary_df["bead_present"].astype(bool)]
        .copy()
        .reset_index(drop=True)
    )
    # Use the stored per-cyst direction summaries written during quantification.
    # theta_moment_deg is currently the selected shell-pixel first-moment
    # definition used by the final Orientation Plot 2, while theta_peak_deg is
    # still helpful for mismatch-style QC ranking in this section.
    bead_ok["theta_peak_error_deg"] = np.abs(
        wrap_angle_deg(bead_ok["theta_peak_deg"] - bead_ok["theta_b_deg"])
    )
    bead_ok["theta_moment_error_deg"] = np.abs(
        wrap_angle_deg(bead_ok["theta_moment_deg"] - bead_ok["theta_b_deg"])
    )
    bead_ok["theta_debug_error_deg"] = np.maximum(
        bead_ok["theta_peak_error_deg"], bead_ok["theta_moment_error_deg"]
    )

    if len(DEBUG_CYST_IDS_MANUAL) > 0:
        diag_ids = [str(c) for c in DEBUG_CYST_IDS_MANUAL]
    else:
        by_d = bead_ok.sort_values("distance_d_um").reset_index(drop=True)
        idx_d = np.unique(
            np.linspace(
                0,
                max(len(by_d) - 1, 0),
                min(N_DEBUG_DISTANCE_EXAMPLES, len(by_d)),
                dtype=int,
            )
        )
        ids_d = by_d.iloc[idx_d]["cyst_id"].astype(str).tolist() if len(by_d) else []

        by_m = bead_ok.sort_values("theta_debug_error_deg", ascending=False)
        ids_m = (
            by_m.head(min(N_DEBUG_MISMATCH_EXAMPLES, len(by_m)))["cyst_id"]
            .astype(str)
            .tolist()
        )

        diag_ids = []
        seen = set()
        for cid in ids_d + ids_m:
            if cid not in seen:
                seen.add(cid)
                diag_ids.append(cid)

    payloads = [build_debug_payload(cid) for cid in diag_ids]

    print("Sampling reminder:")
    print("  Distance plots use the full peripheral shell.")
    print("  Orientation plots use the full peripheral shell.")
    print("Diagnostic cyst IDs:", diag_ids)

    print()
    print("Detailed per-cyst diagnostics:")
    if not RENDER_QC_FIGURES:
        print("Skipping detailed per-cyst figures (RENDER_QC_FIGURES=False).")
    else:
        for payload in payloads:
            cid = str(payload["row"]["cyst_id"])
            print()
            print("-" * 80)
            print(
                f"Cyst diagnostics | {cid} | scene={int(payload['row']['scene_index'])} | "
                f"d={float(payload['row']['distance_d_um']):.1f} um"
            )
            plot_debug_payload(payload)


### Outlier Diagnostics

Start with Distance Plot 1 outliers: bead-present cysts whose full-shell distance traces deviate most from the across-cyst mean +/- SD trend. Plot 2 and orientation outliers are compared later through the candidate sections below.


In [ ]:
# -------------------------------
# Additional diagnostics: Distance Plot 1 outlier deep-dive (still before plots)
# -------------------------------
if OPTIMIZATION_MODE:
    print("Skipping additional diagnostics (OPTIMIZATION_MODE=True).")
else:
    bead_only = summary_df[summary_df["bead_present"].astype(bool)].copy()

    trace_outliers = rank_distance_trace_outliers(
        distance_trace_df,
        summary_df=bead_only,
        n=N_DISTANCE_TRACE_OUTLIERS,
        min_n_cysts_per_bin=TRACE_OUTLIER_MIN_N_CYSTS_PER_BIN,
        sd_floor_quantile=TRACE_OUTLIER_SD_FLOOR_QUANTILE,
    )
    trace_outlier_ids = trace_outliers["cyst_id"].astype(str).tolist()

    print("Distance Plot 1 outlier screen coverage:")
    print(f"  bead-present cysts screened: {len(bead_only)}")
    print(
        "  outlier score: mean absolute deviation from the across-cyst distance trace, scaled by the across-cyst SD at each occupied distance bin"
    )
    print(
        f"  only bins with at least {TRACE_OUTLIER_MIN_N_CYSTS_PER_BIN} contributing cysts are compared; SD floor is the {TRACE_OUTLIER_SD_FLOOR_QUANTILE:g}th percentile of nonzero trace SDs"
    )
    print("Distance Plot 1 outlier cyst IDs:", trace_outlier_ids)
    print()
    print("Plot 2 and orientation outlier screens are deferred for the next pass.")

    if not RENDER_QC_FIGURES:
        print("Skipping Distance Plot 1 outlier figures (RENDER_QC_FIGURES=False).")
    else:
        print()
        print("=" * 80)
        print("Compact DAPI/pSMAD montage of strongest Distance Plot 1 outliers")
        plot_geometry_montage(trace_outlier_ids, ncols=3)

        print()
        print("=" * 80)
        print("Deep-dive panels for strongest Distance Plot 1 outliers")
        for cid in trace_outlier_ids:
            print()
            print("-" * 80)
            print("Distance Plot 1 outlier cyst:", cid)
            outlier_row = trace_outliers.loc[trace_outliers["cyst_id"] == cid].iloc[0]
            plot_distance_trace_outlier_context(
                cid,
                distance_trace_df=distance_trace_df,
                min_n_cysts_per_bin=TRACE_OUTLIER_MIN_N_CYSTS_PER_BIN,
                score_row=outlier_row,
            )
            plot_debug_payload(build_debug_payload(cid))

    print()
    print("Compact Distance Plot 1 outlier reference table:")
    display(
        trace_outliers[
            [
                "cyst_id",
                "scene_index",
                "distance_d_um",
                "ratio_peak",
                "mean_trace_abs_z",
                "rms_trace_z",
                "max_trace_abs_z",
                "n_trace_bins_compared",
                "mean_trace_signed_dev",
                "review_action",
            ]
        ]
    )


## Sensitivity Analyses

These sections compare reasonable analysis choices without changing the final plots below.


### Distance Plot 1 Adaptive Pixel-Support Binning

These diagnostics keep all distance-analysis pixels but merge adjacent exact `10 um` bins until each displayed bin reaches a target pixel support. Individual cyst traces are rebinned independently using that cyst's typical kept-bin support, and the pooled trace is rebinned again across all cysts using the typical pooled support of the exact `10 um` bins. No data are discarded here; sparse regions simply widen.

In [ ]:
# -------------------------------
# Distance Plot 1 adaptive pixel-support diagnostics
# -------------------------------

def _merge_adjacent_exact_distance_bins_by_pixel_support(
    bin_df,
    target_pixels,
    drop_final_under_target=False,
):
    sub = bin_df[["distance_local_um", "ratio_mean", "n_pixels"]].dropna().copy()
    if len(sub) == 0:
        return pd.DataFrame(
            columns=[
                "x_lo",
                "x_hi",
                "width_um",
                "x_center",
                "ratio_mean",
                "n_pixels",
                "n_exact_bins",
            ]
        )
    sub = sub.sort_values("distance_local_um").reset_index(drop=True)
    centers = sub["distance_local_um"].to_numpy(dtype=float)
    values = sub["ratio_mean"].to_numpy(dtype=float)
    weights = sub["n_pixels"].to_numpy(dtype=float)
    positive_weights = weights[np.isfinite(weights) & (weights > 0)]
    if not np.isfinite(target_pixels) or target_pixels <= 0:
        target_pixels = float(np.nanmedian(positive_weights)) if len(positive_weights) else 1.0
    target_pixels = max(float(target_pixels), 1.0)

    rows = []
    i = 0
    half_bin = 0.5 * float(DISTANCE_TRACE_BIN_UM)
    while i < len(sub):
        j = i
        acc = 0.0
        while j < len(sub) and acc < target_pixels:
            acc += float(weights[j])
            j += 1
        xs = centers[i:j]
        ys = values[i:j]
        ws = weights[i:j]
        is_final_chunk = bool(j >= len(sub))
        if drop_final_under_target and is_final_chunk and float(np.sum(ws)) < target_pixels:
            break
        rows.append(
            {
                "x_lo": float(xs[0] - half_bin),
                "x_hi": float(xs[-1] + half_bin),
                "width_um": float(xs[-1] - xs[0] + float(DISTANCE_TRACE_BIN_UM)),
                "x_center": float(np.average(xs, weights=ws)) if np.sum(ws) > 0 else float(np.nanmean(xs)),
                "ratio_mean": float(np.average(ys, weights=ws)) if np.sum(ws) > 0 else float(np.nanmean(ys)),
                "n_pixels": float(np.sum(ws)),
                "n_exact_bins": int(len(xs)),
            }
        )
        i = j
    return pd.DataFrame(rows)


if OPTIMIZATION_MODE:
    print("Skipping Distance Plot 1 adaptive pixel-support diagnostics (OPTIMIZATION_MODE=True).")
else:
    distance_plot1_trace_df = distance_trace_df.copy()
    if "bead_present" in distance_plot1_trace_df.columns:
        distance_plot1_trace_df = distance_plot1_trace_df.loc[
            distance_plot1_trace_df["bead_present"].astype(bool)
        ].copy()

    if len(distance_plot1_trace_df) == 0:
        print("Skipping Distance Plot 1 adaptive pixel-support diagnostics: no bead-present distance traces found.")
    else:
        current_mean_trace = aggregate_trace_across_cysts(
            distance_plot1_trace_df,
            x_col="distance_local_um",
            y_col="ratio_mean",
        )

        pooled_exact_rows = []
        for distance_local_um, sub in distance_plot1_trace_df.groupby("distance_local_um", sort=True):
            weights = sub["n_pixels"].to_numpy(dtype=float)
            pooled_exact_rows.append(
                {
                    "distance_local_um": float(distance_local_um),
                    "ratio_mean": float(np.average(sub["ratio_mean"].to_numpy(dtype=float), weights=weights)),
                    "n_pixels": float(np.sum(weights)),
                    "n_cysts": int(sub["cyst_id"].astype(str).nunique()),
                }
            )
        pooled_exact = pd.DataFrame(pooled_exact_rows)
        pooled_target_pixels = float(np.nanmedian(pooled_exact["n_pixels"].to_numpy(dtype=float)))
        pooled_adaptive = _merge_adjacent_exact_distance_bins_by_pixel_support(
            pooled_exact[["distance_local_um", "ratio_mean", "n_pixels"]],
            pooled_target_pixels,
            drop_final_under_target=True,
        )
        pooled_adaptive["n_cysts"] = [
            int(
                distance_plot1_trace_df.loc[
                    (distance_plot1_trace_df["distance_local_um"] >= row["x_lo"] + 0.5 * DISTANCE_TRACE_BIN_UM)
                    & (distance_plot1_trace_df["distance_local_um"] < row["x_hi"]),
                    "cyst_id",
                ]
                .astype(str)
                .nunique()
            )
            for _, row in pooled_adaptive.iterrows()
        ]

        half_bin = 0.5 * float(DISTANCE_TRACE_BIN_UM)
        adaptive_interval_rows = []
        for _, row in pooled_adaptive.iterrows():
            lo_center = float(row["x_lo"] + half_bin)
            hi_center = float(row["x_hi"] - half_bin)
            interval_sub = distance_plot1_trace_df.loc[
                (distance_plot1_trace_df["distance_local_um"] >= lo_center - 1e-9)
                & (distance_plot1_trace_df["distance_local_um"] <= hi_center + 1e-9)
            ].copy()
            cyst_interval_vals = []
            for _, cyst_sub in interval_sub.groupby("cyst_id", sort=False):
                weights = cyst_sub["n_pixels"].to_numpy(dtype=float)
                cyst_interval_vals.append(
                    float(np.average(cyst_sub["ratio_mean"].to_numpy(dtype=float), weights=weights))
                )
            vals = np.asarray(cyst_interval_vals, dtype=float)
            n = int(np.sum(np.isfinite(vals)))
            mean = float(np.nanmean(vals)) if n else np.nan
            sd = float(np.nanstd(vals, ddof=1)) if n > 1 else (0.0 if n == 1 else np.nan)
            sem = float(sd / np.sqrt(n)) if n > 1 else (0.0 if n == 1 else np.nan)
            adaptive_interval_rows.append(
                {
                    "x_lo": float(row["x_lo"]),
                    "x_hi": float(row["x_hi"]),
                    "x_mid": float(0.5 * (row["x_lo"] + row["x_hi"])),
                    "mean": mean,
                    "sd": sd,
                    "sem": sem,
                    "n_cysts": n,
                    "n_pixels": float(row["n_pixels"]),
                    "n_exact_bins": int(row["n_exact_bins"]),
                }
            )
        adaptive_interval_agg = pd.DataFrame(adaptive_interval_rows)

        summary_table = pd.DataFrame(
            [
                {
                    "trace": "Individual traces",
                    "target rule": "unchanged exact 10 um bins",
                    "target pixels": "not adaptive",
                    "adaptive bins": f"{distance_plot1_trace_df['cyst_id'].astype(str).nunique()} cyst traces",
                    "widths": "10 um exact bins",
                },
                {
                    "trace": "Pooled adaptive trace",
                    "target rule": "median pooled pixel support per exact 10 um bin",
                    "target pixels": f"{pooled_target_pixels:.0f}",
                    "adaptive bins": f"{len(pooled_adaptive)} from {len(pooled_exact)} exact bins",
                    "widths": (
                        f"median width {pooled_adaptive['width_um'].median():.0f} um; "
                        f"max width {pooled_adaptive['width_um'].max():.0f} um"
                    )
                    if len(pooled_adaptive)
                    else "NA",
                },
            ]
        )

        print("Distance Plot 1 adaptive rebinning keeps all exact 10 um bins and merges adjacent bins only until the target pixel support is reached; a final high-distance remainder bin is dropped if it never reaches target support.")
        print("Because each exact bin already stores its pixel count and pixel mean, the merged-bin mean is the exact pixel-weighted mean of the underlying pixels; nothing is approximated beyond the original 10 um binning.")
        print("Gray traces below are the same uniform 10 um individual cyst traces used in the final Distance Plot 1. The red line with points connects the centers of the pooled adaptive bins across all cyst pixels. A second stand-alone plot below shows the adaptive-bin mean +/- SD across cysts as a blue line, styled like the final Distance Plot 1.")
        display(summary_table)
        display(
            pooled_adaptive[["x_lo", "x_hi", "width_um", "n_pixels", "n_exact_bins", "n_cysts"]]
            .rename(columns={"x_lo": "bin_lo_um", "x_hi": "bin_hi_um", "n_cysts": "cysts_spanned"})
            .head(20)
        )

        if not RENDER_SENSITIVITY_ANALYSIS_FIGURES:
            print("Skipping Distance Plot 1 adaptive pixel-support figures (RENDER_SENSITIVITY_ANALYSIS_FIGURES=False).")
        else:
            from matplotlib.lines import Line2D

            xvals_all = distance_plot1_trace_df["distance_local_um"].to_numpy(dtype=float)
            xlim = (
                float(np.nanmin(xvals_all) - 0.5 * DISTANCE_TRACE_BIN_UM),
                float(np.nanmax(xvals_all) + 0.5 * DISTANCE_TRACE_BIN_UM),
            )
            ymins = []
            ymaxs = []
            if len(distance_plot1_trace_df):
                ymins.append(float(np.nanmin(distance_plot1_trace_df["ratio_mean"])))
                ymaxs.append(float(np.nanmax(distance_plot1_trace_df["ratio_mean"])))
            if len(current_mean_trace):
                ymins.append(float(np.nanmin(current_mean_trace["mean"] - current_mean_trace["sd"])))
                ymaxs.append(float(np.nanmax(current_mean_trace["mean"] + current_mean_trace["sd"])))
            if len(pooled_adaptive):
                ymins.append(float(np.nanmin(pooled_adaptive["ratio_mean"])))
                ymaxs.append(float(np.nanmax(pooled_adaptive["ratio_mean"])))
            ypad = 0.06 * max(1e-6, max(ymaxs) - min(ymins))
            ylim = (min(ymins) - ypad, max(ymaxs) + ypad)

            fig = plt.figure(figsize=(14.2, 8.8), constrained_layout=True)
            gs = fig.add_gridspec(2, 1, height_ratios=[3.5, 1.7], hspace=0.12)
            ax = fig.add_subplot(gs[0])
            ax_sup = fig.add_subplot(gs[1], sharex=ax)

            for _, sub in distance_plot1_trace_df.groupby("cyst_id"):
                sub = sub.sort_values("distance_local_um")
                ax.plot(
                    sub["distance_local_um"],
                    sub["ratio_mean"],
                    color="0.65",
                    alpha=0.14,
                    linewidth=0.8,
                )
                ax.scatter(
                    sub["distance_local_um"],
                    sub["ratio_mean"],
                    color="0.65",
                    alpha=0.09,
                    s=6,
                )

            if len(current_mean_trace):
                ax.plot(
                    current_mean_trace["distance_local_um"],
                    current_mean_trace["mean"],
                    color="tab:blue",
                    linewidth=2.0,
                    label="Current fixed-bin mean trace",
                )

            if len(pooled_adaptive):
                ax.plot(
                    pooled_adaptive["x_center"],
                    pooled_adaptive["ratio_mean"],
                    color="crimson",
                    linewidth=1.8,
                    marker="o",
                    markersize=3.8,
                    alpha=0.95,
                    label="Adaptive pooled-pixel trace",
                )

            ax.set_title("Distance Plot 1 with adaptive pixel-support binning")
            ax.set_ylabel("Normalized pSMAD/DAPI")
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)
            ax.grid(alpha=0.12, linewidth=0.6)
            ax.tick_params(labelbottom=False)
            ax.legend(
                handles=[
                    Line2D([0], [0], color="0.65", alpha=0.55, linewidth=1.2, label="Uniform individual traces"),
                    Line2D([0], [0], color="tab:blue", linewidth=2.0, label="Current fixed-bin mean trace"),
                    Line2D([0], [0], color="crimson", linewidth=2.4, label="Adaptive pooled-pixel trace"),
                ],
                loc="best",
                frameon=False,
            )

            if len(pooled_adaptive):
                ax_sup.bar(
                    pooled_adaptive["x_lo"].to_numpy(dtype=float),
                    pooled_adaptive["n_pixels"].to_numpy(dtype=float),
                    width=(pooled_adaptive["x_hi"] - pooled_adaptive["x_lo"]).to_numpy(dtype=float),
                    align="edge",
                    color="crimson",
                    alpha=0.28,
                    edgecolor="crimson",
                    linewidth=0.4,
                    label="Pooled adaptive bins",
                )
                ax_sup.axhline(
                    pooled_target_pixels,
                    color="crimson",
                    linestyle="--",
                    linewidth=1.2,
                    label=f"Target support ({pooled_target_pixels:.0f} pixels)",
                )
                ax_sup2 = ax_sup.twinx()
                ax_sup2.plot(
                    pooled_adaptive["x_center"].to_numpy(dtype=float),
                    pooled_adaptive["width_um"].to_numpy(dtype=float),
                    color="0.25",
                    marker="o",
                    markersize=2.8,
                    linewidth=1.2,
                    label="Adaptive bin width",
                )
                ax_sup2.set_ylabel("Adaptive bin width (um)", color="0.25")
                ax_sup2.tick_params(axis="y", colors="0.25")
                h1, l1 = ax_sup.get_legend_handles_labels()
                h2, l2 = ax_sup2.get_legend_handles_labels()
                ax_sup.legend(h1 + h2, l1 + l2, loc="upper right", frameon=False, fontsize=8)
            ax_sup.set_title("Pooled adaptive bins: bar height = total pixels, bar width = distance span")
            ax_sup.set_ylabel("Total pixels/bin")
            ax_sup.set_xlabel("Local bead-to-shell-pixel distance (um)")
            ax_sup.set_xlim(*xlim)
            ax_sup.grid(axis="y", alpha=0.12, linewidth=0.6)

            plt.show()

            fig_alt, ax_alt = plt.subplots(1, 1, figsize=(7.2, 5.8))
            for _, sub in distance_plot1_trace_df.groupby("cyst_id"):
                sub = sub.sort_values("distance_local_um")
                ax_alt.plot(
                    sub["distance_local_um"],
                    sub["ratio_mean"],
                    color="0.65",
                    alpha=0.14,
                    linewidth=0.8,
                )
                ax_alt.scatter(
                    sub["distance_local_um"],
                    sub["ratio_mean"],
                    color="0.65",
                    alpha=0.09,
                    s=6,
                )
            if len(adaptive_interval_agg):
                ax_alt.plot(
                    adaptive_interval_agg["x_mid"],
                    adaptive_interval_agg["mean"],
                    color="tab:blue",
                    linewidth=2.6,
                    label="Mean trace",
                )
                ax_alt.fill_between(
                    adaptive_interval_agg["x_mid"],
                    adaptive_interval_agg["mean"] - adaptive_interval_agg["sd"],
                    adaptive_interval_agg["mean"] + adaptive_interval_agg["sd"],
                    color="tab:blue",
                    alpha=0.22,
                    label="±SD",
                )
            ax_alt.set_xlabel("Local bead-to-shell-pixel distance (um)")
            ax_alt.set_ylabel("Normalized pSMAD/DAPI")
            ax_alt.set_title("Distance Plot 1 alternative: adaptive pooled-support bins")
            ax_alt.legend(loc="best")
            ax_alt.set_xlim(*xlim)
            ax_alt.set_ylim(*ylim)
            ax_alt.grid(alpha=0.12, linewidth=0.6)
            plt.show()

### Distance Plot 2 Candidates

Compare alternative cyst-level response summaries for Distance Plot 2 under the same piecewise responsive-regime fit used for the final plot.


In [ ]:
# -------------------------------
# Distance Plot 2 candidate summary diagnostics
# -------------------------------
bead_summary = summary_df[summary_df["bead_present"].astype(bool)].copy()
control_summary = summary_df[~summary_df["bead_present"].astype(bool)].copy()

PLOT2_COMPARABLE_METRICS = [
    ("ratio_raw_max", "Raw max"),
    ("ratio_p99", "99th percentile"),
    ("ratio_p95", "95th percentile"),
    ("ratio_p90", "90th percentile"),
    ("ratio_peak_top01", "Top 1% mean"),
    ("ratio_peak_top05", "Top 5% mean"),
    ("ratio_peak", "Top 10% mean (legacy)"),
    ("ratio_peak_top20", "Top 20% mean"),
    ("ratio_peak_top30", "Top 30% mean"),
    ("ratio_mean_shell", "Mean of full shell"),
    ("ratio_median_shell", "Median of full shell"),
    ("ratio_mean_distance_region", "Mean of kept shell region"),
    ("ratio_median_distance_region", "Median of kept shell region"),
]
PLOT2_TARGETED_METRICS = [
    ("ratio_near_q05_mean", "Nearest 5% of distance pixels"),
    ("ratio_near_q10_mean", "Nearest 10% of distance pixels"),
    ("ratio_near_q20_mean", "Nearest 20% of distance pixels"),
    ("ratio_near_q30_mean", "Nearest 30% of distance pixels"),
    ("ratio_bead_crescent_30_mean", "Bead-facing +/-30 deg"),
    ("ratio_bead_crescent_45_mean", "Bead-facing +/-45 deg"),
    ("ratio_bead_crescent_60_mean", "Bead-facing +/-60 deg"),
    ("ratio_bead_crescent_90_mean", "Bead-facing +/-90 deg"),
]
PLOT2_RESPONSIVE_MIN_POINTS = 20
PLOT2_RESPONSIVE_MIN_FRACTION = 0.15
PLOT2_CANDIDATE_FIT_CSV = OUT_DIR / "manual_gradient_plot2_candidate_fits.csv"


def _linear_slope_pvalue(x_vals, y_vals):
    d = pd.DataFrame({"x": x_vals, "y": y_vals}).dropna().copy()
    if len(d) < 3 or d["x"].nunique() < 2:
        return np.nan
    fit = stats.linregress(d["x"].to_numpy(dtype=float), d["y"].to_numpy(dtype=float))
    return float(fit.pvalue) if np.isfinite(fit.pvalue) else np.nan


def _fit_plot2_responsive_regime(x, y, min_points=20, min_fraction=0.15):
    d = pd.DataFrame({"x": x, "y": y}).dropna().copy()
    if len(d) < 6:
        return {"valid": False}
    d["_orig_idx"] = np.arange(len(d), dtype=int)
    d = d.sort_values("x").reset_index(drop=True)
    x = d["x"].to_numpy(dtype=float)
    y = d["y"].to_numpy(dtype=float)
    orig_idx = d["_orig_idx"].to_numpy(dtype=int)
    n = len(d)
    min_n = max(int(min_points), int(np.ceil(float(min_fraction) * n)))
    min_n = min(min_n, max(2, n // 3))
    if n < 2 * min_n + 1:
        return {"valid": False}

    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    best_any = None
    best_neg = None
    for split in range(min_n, n - min_n + 1):
        x_left = x[:split]
        y_left = y[:split]
        x_right = x[split:]
        y_right = y[split:]
        responsive_slope, responsive_intercept, responsive_r2 = linear_fit_r2(x_left, y_left)
        if not np.isfinite(responsive_slope) or not np.isfinite(responsive_intercept):
            continue
        plateau_mean = float(np.nanmean(y_right))
        plateau_lin_slope, plateau_lin_intercept, plateau_lin_r2 = linear_fit_r2(x_right, y_right)
        plateau_flat_p = _linear_slope_pvalue(x_right, y_right)
        yhat_left = responsive_slope * x_left + responsive_intercept
        sse_left = float(np.nansum((y_left - yhat_left) ** 2))
        sse_right = float(np.nansum((y_right - plateau_mean) ** 2))
        sse_total = sse_left + sse_right
        cutoff_um = float(0.5 * (x[split - 1] + x[split]))
        endpoint_y = float(responsive_slope * cutoff_um + responsive_intercept)
        rho_left = pd.Series(x_left).corr(pd.Series(y_left), method="spearman")
        rho_right = pd.Series(x_right).corr(pd.Series(y_right), method="spearman")
        piecewise_r2 = np.nan if ss_tot <= 0 else float(1.0 - sse_total / ss_tot)
        cand = {
            "valid": True,
            "cutoff_um": cutoff_um,
            "responsive_slope": float(responsive_slope),
            "responsive_intercept": float(responsive_intercept),
            "responsive_r2": float(responsive_r2),
            "responsive_rho": float(rho_left) if np.isfinite(rho_left) else np.nan,
            "piecewise_r2": piecewise_r2,
            "plateau_mean": plateau_mean,
            "endpoint_y": endpoint_y,
            "responsive_n": int(split),
            "plateau_n": int(len(y_right)),
            "sse_total": sse_total,
            "x_left_min": float(np.nanmin(x_left)),
            "x_left_max": float(np.nanmax(x_left)),
            "plateau_x_min": float(np.nanmin(x_right)),
            "plateau_x_max": float(np.nanmax(x_right)),
            "x_all_max": float(np.nanmax(x)),
            "responsive_idx": orig_idx[:split].copy(),
            "plateau_idx": orig_idx[split:].copy(),
            "plateau_linear_slope": float(plateau_lin_slope) if np.isfinite(plateau_lin_slope) else np.nan,
            "plateau_linear_intercept": float(plateau_lin_intercept) if np.isfinite(plateau_lin_intercept) else np.nan,
            "plateau_linear_r2": float(plateau_lin_r2) if np.isfinite(plateau_lin_r2) else np.nan,
            "plateau_linear_rho": float(rho_right) if np.isfinite(rho_right) else np.nan,
            "plateau_flat_p": float(plateau_flat_p) if np.isfinite(plateau_flat_p) else np.nan,
        }
        if best_any is None or cand["sse_total"] < best_any["sse_total"]:
            best_any = cand
        if responsive_slope < 0 and (best_neg is None or cand["sse_total"] < best_neg["sse_total"]):
            best_neg = cand
    best = best_neg if best_neg is not None else best_any
    return best if best is not None else {"valid": False}


def _plot_plot2_candidate_grid(metrics, title, include_controls=True, ncols=4, render=True):
    n = len(metrics)
    nrows = int(np.ceil(n / ncols))
    if render:
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.7 * ncols, 4.0 * nrows), constrained_layout=True)
        axes = np.atleast_1d(axes).ravel()
    else:
        fig = None
        axes = [None] * n
    summary_rows = []
    first_ax = True
    for ax, (metric_col, metric_label) in zip(axes, metrics):
        bead_sub = bead_summary[["distance_d_um", metric_col]].dropna().copy()
        x = bead_sub["distance_d_um"].to_numpy(dtype=float)
        y = bead_sub[metric_col].to_numpy(dtype=float)
        regime = _fit_plot2_responsive_regime(
            x,
            y,
            min_points=PLOT2_RESPONSIVE_MIN_POINTS,
            min_fraction=PLOT2_RESPONSIVE_MIN_FRACTION,
        )

        if render:
            ax.scatter(
                x,
                y,
                s=22,
                alpha=0.52,
                color="tab:blue",
                label="Bead-present cysts" if first_ax else None,
            )
        if regime.get("valid", False) and render:
            x_left = np.linspace(regime["x_left_min"], regime["cutoff_um"], 100)
            y_left = regime["responsive_slope"] * x_left + regime["responsive_intercept"]
            ax.plot(
                x_left,
                y_left,
                color="crimson",
                linestyle="--",
                linewidth=1.6,
                label="Responsive-regime fit" if first_ax else None,
            )
            ax.axvline(
                regime["cutoff_um"],
                color="crimson",
                linestyle=":",
                linewidth=1.2,
                label="Responsive cutoff" if first_ax else None,
            )
            ax.hlines(
                regime["plateau_mean"],
                regime["cutoff_um"],
                regime["x_all_max"],
                colors="tab:orange",
                linestyles="-.",
                linewidth=1.4,
                label="Plateau mean" if first_ax else None,
            )
            ax.axvspan(regime["x_left_min"], regime["cutoff_um"], color="crimson", alpha=0.04)

        control_note = ""
        if include_controls and metric_col in control_summary.columns:
            ctrl = control_summary[["distance_plot_um", metric_col]].dropna().copy()
            if len(ctrl) > 0:
                cx = ctrl["distance_plot_um"].to_numpy(dtype=float)
                cy = ctrl[metric_col].to_numpy(dtype=float)
                if render:
                    ax.scatter(
                        cx,
                        cy,
                        marker="s",
                        s=34,
                        alpha=0.90,
                        color="0.35",
                        label="No-bead controls" if first_ax else None,
                    )
                cmean = float(np.nanmean(cy))
                csd = float(np.nanstd(cy, ddof=1)) if len(cy) > 1 else 0.0
                xmin = float(np.nanmin(x)) if len(x) else float(np.nanmin(cx))
                xmax = max(float(np.nanmax(x)) if len(x) else xmin, float(np.nanmax(cx)))
                if render:
                    ax.axhline(
                        cmean,
                        color="0.35",
                        linestyle=":",
                        linewidth=1.1,
                        label="No-bead mean" if first_ax else None,
                    )
                    if np.isfinite(csd) and csd > 0:
                        ax.fill_between([xmin, xmax], [cmean - csd, cmean - csd], [cmean + csd, cmean + csd], color="0.55", alpha=0.08)
                control_note = " | ctrl"

        if regime.get("valid", False):
            resp_r2_txt = f"{regime['responsive_r2']:.2f}" if np.isfinite(regime['responsive_r2']) else "NA"
            resp_rho_txt = f"{regime['responsive_rho']:.2f}" if np.isfinite(regime['responsive_rho']) else "NA"
            plat_p_txt = f"{regime['plateau_flat_p']:.2g}" if np.isfinite(regime['plateau_flat_p']) else "NA"
            cut_txt = f"{regime['cutoff_um']:.0f}"
            title_txt = (
                f"{metric_label}\n"
                f"cut={cut_txt} um | resp R^2={resp_r2_txt} | resp rho={resp_rho_txt}\n"
                f"plateau nested p(flat vs line)={plat_p_txt}{control_note}"
            )
            summary_rows.append({
                "metric_col": metric_col,
                "metric_label": metric_label,
                "responsive_cutoff_um": regime["cutoff_um"],
                "responsive_r2": regime["responsive_r2"],
                "responsive_rho": regime["responsive_rho"],
                "piecewise_r2": regime["piecewise_r2"],
                "responsive_n": regime["responsive_n"],
                "plateau_n": regime["plateau_n"],
                "plateau_mean": regime["plateau_mean"],
                "plateau_linear_r2": regime["plateau_linear_r2"],
                "plateau_linear_rho": regime["plateau_linear_rho"],
                "plateau_linear_slope": regime["plateau_linear_slope"],
                "plateau_flat_p": regime["plateau_flat_p"],
                "control_comparable": bool(include_controls and metric_col in control_summary.columns),
            })
        else:
            title_txt = f"{metric_label}\nno stable responsive regime{control_note}"
            summary_rows.append({
                "metric_col": metric_col,
                "metric_label": metric_label,
                "responsive_cutoff_um": np.nan,
                "responsive_r2": np.nan,
                "responsive_rho": np.nan,
                "piecewise_r2": np.nan,
                "responsive_n": np.nan,
                "plateau_n": np.nan,
                "plateau_mean": np.nan,
                "plateau_linear_r2": np.nan,
                "plateau_linear_rho": np.nan,
                "plateau_linear_slope": np.nan,
                "plateau_flat_p": np.nan,
                "control_comparable": bool(include_controls and metric_col in control_summary.columns),
            })
        if render:
            ax.set_title(title_txt, fontsize=10)
            ax.set_xlabel("Nearest bead-center to cyst-surface distance d (um)")
            ax.set_ylabel("Candidate cyst-level pSMAD/DAPI summary")
            if first_ax:
                ax.legend(loc="best", fontsize=7.5, frameon=False)
                first_ax = False

    if render:
        for ax in axes[n:]:
            ax.axis("off")
        fig.suptitle(title, fontsize=12, y=1.01)
        plt.show()
    return pd.DataFrame(summary_rows)

print("Distance Plot 2 diagnostics: compare many candidate cyst-level response summaries before changing the main plot.")
print("Responsive regime is chosen objectively per metric by best piecewise least-squares fit: left linear segment + right plateau.")
print("Raw points are shown directly, and panel titles report the responsive-section fit plus a nested flat-vs-line p-value for the distal subset.")

plot2_candidate_fit_df = pd.DataFrame()
if OPTIMIZATION_MODE:
    print("Skipping Distance Plot 2 candidate-summary figures (OPTIMIZATION_MODE=True).")
else:
    render_sensitivity_figures = bool(RENDER_SENSITIVITY_ANALYSIS_FIGURES)
    if not render_sensitivity_figures:
        print("Skipping Distance Plot 2 candidate-summary figures (RENDER_SENSITIVITY_ANALYSIS_FIGURES=False).")
    fit_df_a = _plot_plot2_candidate_grid(
        PLOT2_COMPARABLE_METRICS,
        title="Distance Plot 2 candidates: control-comparable response summaries",
        include_controls=True,
        ncols=4,
        render=render_sensitivity_figures,
    )
    fit_df_a["family"] = "control_comparable"
    fit_df_b = _plot_plot2_candidate_grid(
        PLOT2_TARGETED_METRICS,
        title="Distance Plot 2 candidates: bead-targeted response summaries",
        include_controls=False,
        ncols=4,
        render=render_sensitivity_figures,
    )
    fit_df_b["family"] = "bead_targeted"
    plot2_candidate_fit_df = pd.concat([fit_df_a, fit_df_b], ignore_index=True)
    plot2_candidate_fit_df.to_csv(PLOT2_CANDIDATE_FIT_CSV, index=False)
    print("Saved Plot 2 candidate fit summary:", PLOT2_CANDIDATE_FIT_CSV)






### Orientation Plot 1 Gate Sensitivity

Check how the aggregate orientation trace changes as the bead-responsive distance gate becomes stricter. The final Orientation Plot 1 still uses the standard Plot 2-derived gate.


In [ ]:
# -------------------------------
# Orientation Plot 1 candidate distance-gate diagnostics
# -------------------------------
bead_summary = summary_df[summary_df["bead_present"].astype(bool)].copy()
plot2_gate_regime = _fit_plot2_responsive_regime(
    bead_summary["distance_d_um"].to_numpy(dtype=float),
    bead_summary[PLOT2_STANDARD_METRIC_COL].to_numpy(dtype=float),
    min_points=PLOT2_RESPONSIVE_MIN_POINTS,
    min_fraction=PLOT2_RESPONSIVE_MIN_FRACTION,
)

if OPTIMIZATION_MODE:
    print("Skipping Orientation Plot 1 candidate gate diagnostics (OPTIMIZATION_MODE=True).")
elif len(bead_summary) == 0 or len(orientation_trace_df) == 0:
    print("Skipping Orientation Plot 1 candidate gate diagnostics: missing bead-present summary rows or orientation traces.")
elif not plot2_gate_regime.get("valid", False):
    print("Skipping Orientation Plot 1 candidate gate diagnostics: no stable Plot 2 responsive cutoff available.")
else:
    base_cutoff_um = float(plot2_gate_regime["cutoff_um"])
    gate_specs = [
        ("All bead-present", np.inf),
        (f"Current gate\nd <= {base_cutoff_um:.0f} um", base_cutoff_um),
        (f"95% of current\nd <= {0.95 * base_cutoff_um:.0f} um", 0.95 * base_cutoff_um),
        (f"90% of current\nd <= {0.90 * base_cutoff_um:.0f} um", 0.90 * base_cutoff_um),
        (f"85% of current\nd <= {0.85 * base_cutoff_um:.0f} um", 0.85 * base_cutoff_um),
        (f"80% of current\nd <= {0.80 * base_cutoff_um:.0f} um", 0.80 * base_cutoff_um),
    ]

    candidate_results = []
    for label, cutoff_um in gate_specs:
        if np.isfinite(cutoff_um):
            keep_ids = set(
                bead_summary.loc[bead_summary["distance_d_um"] <= cutoff_um, "cyst_id"].dropna().astype(str)
            )
        else:
            keep_ids = set(bead_summary["cyst_id"].dropna().astype(str))
        trace_sub = orientation_trace_df[
            orientation_trace_df["cyst_id"].astype(str).isin(keep_ids)
        ].copy()
        agg = aggregate_trace_across_cysts(
            trace_sub,
            x_col="theta_rel_bead_deg",
            y_col="ratio_norm_mean",
        )
        candidate_results.append({
            "label": label,
            "cutoff_um": cutoff_um,
            "n_cysts": int(trace_sub["cyst_id"].astype(str).nunique()),
            "trace_sub": trace_sub,
            "agg": agg,
        })

    ymins = []
    ymaxs = []
    for res in candidate_results:
        agg = res["agg"]
        if len(agg) > 0:
            ymins.append(float(np.nanmin(agg["mean"] - agg["sd"])))
            ymaxs.append(float(np.nanmax(agg["mean"] + agg["sd"])))
    if ymins and ymaxs:
        ypad = 0.05 * max(1e-6, max(ymaxs) - min(ymins))
        ylim = (min(ymins) - ypad, max(ymaxs) + ypad)
    else:
        ylim = None

    print("Orientation Plot 1 candidate gates are diagnostic only.")
    print("Final Orientation Plot 1 below still uses the current Plot 2 responsive cutoff.")
    print("Current responsive cutoff for reference:", f"{base_cutoff_um:.1f} um")

    if not RENDER_SENSITIVITY_ANALYSIS_FIGURES:
        print("Skipping Orientation Plot 1 candidate gate figures (RENDER_SENSITIVITY_ANALYSIS_FIGURES=False).")
    else:
        fig, axes = plt.subplots(2, 3, figsize=(15.6, 8.8), sharex=True, sharey=True)
        axes = np.atleast_1d(axes).ravel()
        for ax, res in zip(axes, candidate_results):
            agg = res["agg"]
            if len(agg) > 0:
                ax.plot(
                    agg["theta_rel_bead_deg"],
                    agg["mean"],
                    color="tab:green",
                    linewidth=2.3,
                )
                ax.fill_between(
                    agg["theta_rel_bead_deg"],
                    agg["mean"] - agg["sd"],
                    agg["mean"] + agg["sd"],
                    color="tab:green",
                    alpha=0.22,
                )
            ax.axvline(0.0, color="red", linestyle="--", linewidth=1.3)
            ax.set_title(f"{res['label']}\n{res['n_cysts']} cysts", fontsize=10)
            ax.set_xlabel("Angle relative to bead direction (deg)")
            ax.set_ylabel("Cyst-normalized bg-normalized pSMAD/DAPI")
            ax.set_xlim(-180, 180)
            if ylim is not None:
                ax.set_ylim(*ylim)
        for ax in axes[len(candidate_results):]:
            ax.axis("off")
        fig.suptitle("Orientation Plot 1 sensitivity to distance-gate cutoff", fontsize=12, y=1.01)
        plt.tight_layout()
        plt.show()



### Orientation Plot 2 Candidates

Compare alternative cyst-level orientation summaries under the same bead-responsive distance gate used in the final Orientation Plot 2.


In [ ]:
# -------------------------------
# Orientation Plot 2 candidate direction-summary diagnostics
# -------------------------------
bead_summary = summary_df[summary_df["bead_present"].astype(bool)].copy()
plot2_gate_regime = _fit_plot2_responsive_regime(
    bead_summary["distance_d_um"].to_numpy(dtype=float),
    bead_summary[PLOT2_STANDARD_METRIC_COL].to_numpy(dtype=float),
    min_points=PLOT2_RESPONSIVE_MIN_POINTS,
    min_fraction=PLOT2_RESPONSIVE_MIN_FRACTION,
)
ORIENTATION_PLOT2_CANDIDATE_FIT_CSV = OUT_DIR / "manual_gradient_orientation_plot2_candidate_fits.csv"


def _circular_weighted_mean_deg(theta_deg, weights):
    theta_deg = np.asarray(theta_deg, dtype=float)
    weights = np.asarray(weights, dtype=float)
    keep = np.isfinite(theta_deg) & np.isfinite(weights) & (weights > 0)
    if np.sum(keep) == 0:
        return np.nan
    ang = np.deg2rad(theta_deg[keep])
    w = weights[keep]
    vx = np.sum(w * np.cos(ang))
    vy = np.sum(w * np.sin(ang))
    if not np.isfinite(vx) or not np.isfinite(vy) or (abs(vx) < 1e-12 and abs(vy) < 1e-12):
        return np.nan
    return float(np.rad2deg(np.arctan2(vy, vx)))


def _peak_direction_from_trace(rel_deg, values, theta_b_deg, smooth_bins=None):
    rel_deg = np.asarray(rel_deg, dtype=float)
    values = np.asarray(values, dtype=float)
    keep = np.isfinite(rel_deg) & np.isfinite(values)
    if np.sum(keep) == 0:
        return np.nan
    rel_deg = rel_deg[keep]
    values = values[keep]
    eval_vals = circular_smooth(values, window_bins=smooth_bins) if smooth_bins and smooth_bins > 1 else values
    if not np.any(np.isfinite(eval_vals)):
        return np.nan
    idx = int(np.nanargmax(eval_vals))
    return float(wrap_angle_deg(rel_deg[idx] + theta_b_deg))


def _topk_bin_circular_mean_deg(rel_deg, values, theta_b_deg, k):
    rel_deg = np.asarray(rel_deg, dtype=float)
    values = np.asarray(values, dtype=float)
    keep = np.isfinite(rel_deg) & np.isfinite(values)
    if np.sum(keep) == 0:
        return np.nan
    rel_deg = rel_deg[keep]
    values = values[keep]
    k = int(min(max(1, k), len(values)))
    idx = np.argsort(values)[-k:]
    abs_deg = wrap_angle_deg(rel_deg[idx] + theta_b_deg)
    weights = np.clip(values[idx], 0.0, None)
    return _circular_weighted_mean_deg(abs_deg, weights)


def _moment_direction_from_trace(rel_deg, values, counts, theta_b_deg, baseline_percentile):
    rel_deg = np.asarray(rel_deg, dtype=float)
    values = np.asarray(values, dtype=float)
    counts = np.asarray(counts, dtype=float)
    keep = np.isfinite(rel_deg) & np.isfinite(values) & np.isfinite(counts) & (counts > 0)
    if np.sum(keep) == 0:
        return np.nan
    rel_deg = rel_deg[keep]
    values = values[keep]
    counts = counts[keep]
    baseline = 0.0 if baseline_percentile <= 0 else float(np.nanpercentile(values, baseline_percentile))
    weights = np.clip(values - baseline, 0.0, None) * counts
    if np.nansum(weights) <= 0:
        return np.nan
    abs_deg = wrap_angle_deg(rel_deg + theta_b_deg)
    return _circular_weighted_mean_deg(abs_deg, weights)


def _median_abs_angle_error_deg(x_deg, y_deg):
    x_deg = np.asarray(x_deg, dtype=float)
    y_deg = np.asarray(y_deg, dtype=float)
    keep = np.isfinite(x_deg) & np.isfinite(y_deg)
    if np.sum(keep) == 0:
        return np.nan
    return float(np.nanmedian(np.abs(wrap_angle_deg(y_deg[keep] - x_deg[keep]))))


if OPTIMIZATION_MODE:
    print("Skipping Orientation Plot 2 candidate diagnostics (OPTIMIZATION_MODE=True).")
elif len(bead_summary) == 0 or len(orientation_trace_df) == 0:
    print("Skipping Orientation Plot 2 candidate diagnostics: missing bead-present summary rows or orientation traces.")
elif not plot2_gate_regime.get("valid", False):
    print("Skipping Orientation Plot 2 candidate diagnostics: no stable Plot 2 responsive cutoff available.")
else:
    base_cutoff_um = float(plot2_gate_regime["cutoff_um"])
    gated_summary = bead_summary.loc[
        bead_summary["distance_d_um"] <= base_cutoff_um
    ].copy()
    gated_ids = set(gated_summary["cyst_id"].dropna().astype(str))
    gated_trace = orientation_trace_df.loc[
        orientation_trace_df["cyst_id"].astype(str).isin(gated_ids)
    ].copy()

    candidate_rows = []
    for cid, sub in gated_trace.groupby("cyst_id"):
        hit = gated_summary.loc[gated_summary["cyst_id"].astype(str) == str(cid)]
        if len(hit) == 0:
            continue
        row = hit.iloc[0]
        rel_deg = sub["theta_rel_bead_deg"].to_numpy(dtype=float)
        trace_vals = sub["ratio_norm_mean"].to_numpy(dtype=float)
        counts = sub["n_pixels"].to_numpy(dtype=float)
        theta_b = float(row["theta_b_deg"])
        candidate_rows.append(
            {
                "cyst_id": str(cid),
                "theta_b_deg": theta_b,
                "theta_peak_stored_deg": float(row["theta_peak_deg"]),
                "theta_peak_raw_bin_deg": _peak_direction_from_trace(rel_deg, trace_vals, theta_b, smooth_bins=None),
                "theta_peak_smooth3_deg": _peak_direction_from_trace(rel_deg, trace_vals, theta_b, smooth_bins=3),
                "theta_peak_smooth5_deg": _peak_direction_from_trace(rel_deg, trace_vals, theta_b, smooth_bins=5),
                "theta_peak_smooth7_deg": _peak_direction_from_trace(rel_deg, trace_vals, theta_b, smooth_bins=7),
                "theta_peak_top3_circmean_deg": _topk_bin_circular_mean_deg(rel_deg, trace_vals, theta_b, k=3),
                "theta_peak_top5_circmean_deg": _topk_bin_circular_mean_deg(rel_deg, trace_vals, theta_b, k=5),
                "theta_peak_top7_circmean_deg": _topk_bin_circular_mean_deg(rel_deg, trace_vals, theta_b, k=7),
                "theta_moment_stored_deg": float(row["theta_moment_deg"]),
                "theta_moment_trace_b0_deg": _moment_direction_from_trace(rel_deg, trace_vals, counts, theta_b, baseline_percentile=0.0),
                "theta_moment_trace_b10_deg": _moment_direction_from_trace(rel_deg, trace_vals, counts, theta_b, baseline_percentile=10.0),
                "theta_moment_trace_b20_deg": _moment_direction_from_trace(rel_deg, trace_vals, counts, theta_b, baseline_percentile=20.0),
                "theta_moment_trace_b30_deg": _moment_direction_from_trace(rel_deg, trace_vals, counts, theta_b, baseline_percentile=30.0),
                "theta_moment_trace_b40_deg": _moment_direction_from_trace(rel_deg, trace_vals, counts, theta_b, baseline_percentile=40.0),
            }
        )

    candidate_df = pd.DataFrame(candidate_rows)
    peak_specs = [
        ("theta_peak_stored_deg", "Peak bin of 5-bin-smoothed\nfull angular trace"),
        ("theta_peak_raw_bin_deg", "Peak bin of\nraw angular trace"),
        ("theta_peak_smooth3_deg", "Peak bin of\n3-bin-smoothed trace"),
        ("theta_peak_smooth5_deg", "Peak bin of 5-bin-smoothed\noccupied-bin trace"),
        ("theta_peak_smooth7_deg", "Peak bin of\n7-bin-smoothed trace"),
        ("theta_peak_top3_circmean_deg", "Circular mean of\nstrongest 3 bins"),
        ("theta_peak_top5_circmean_deg", "Circular mean of\nstrongest 5 bins"),
        ("theta_peak_top7_circmean_deg", "Circular mean of\nstrongest 7 bins"),
    ]
    moment_specs = [
        ("theta_moment_stored_deg", "Shell-pixel first moment\n(20th-pct baseline)"),
        ("theta_moment_trace_b0_deg", "Trace-binned first moment\n(0th-pct baseline)"),
        ("theta_moment_trace_b10_deg", "Trace-binned first moment\n(10th-pct baseline)"),
        ("theta_moment_trace_b20_deg", "Trace-binned first moment\n(20th-pct baseline)"),
        ("theta_moment_trace_b30_deg", "Trace-binned first moment\n(30th-pct baseline)"),
        ("theta_moment_trace_b40_deg", "Trace-binned first moment\n(40th-pct baseline)"),
    ]

    fit_rows = []
    for group_name, specs in [("peak_like", peak_specs), ("moment_like", moment_specs)]:
        for col, label in specs:
            sub = candidate_df[["theta_b_deg", col]].dropna().copy()
            x = sub["theta_b_deg"].to_numpy(dtype=float)
            y = sub[col].to_numpy(dtype=float)
            slope, intercept, r2 = linear_fit_r2(x, y)
            rho = pd.Series(x).corr(pd.Series(y), method="spearman")
            fit_rows.append(
                {
                    "group": group_name,
                    "metric_col": col,
                    "label": label.replace("\n", " "),
                    "n_cysts": int(len(sub)),
                    "slope": float(slope) if np.isfinite(slope) else np.nan,
                    "intercept": float(intercept) if np.isfinite(intercept) else np.nan,
                    "r2": float(r2) if np.isfinite(r2) else np.nan,
                    "rho": float(rho) if np.isfinite(rho) else np.nan,
                    "median_abs_angle_error_deg": _median_abs_angle_error_deg(x, y),
                }
            )
    fit_df = pd.DataFrame(fit_rows)
    fit_df.to_csv(ORIENTATION_PLOT2_CANDIDATE_FIT_CSV, index=False)

    print("Orientation Plot 2 candidate statistics are diagnostic only.")
    print("They use the same responsive-cyst distance gate as the final orientation plots.")
    print("Current responsive cutoff for reference:", f"{base_cutoff_um:.1f} um")
    print("Responsive cysts included:", int(candidate_df["cyst_id"].astype(str).nunique()), "/", int(bead_summary["cyst_id"].astype(str).nunique()))
    print("Candidate fit summary CSV:", ORIENTATION_PLOT2_CANDIDATE_FIT_CSV)
    print("Candidate panel key:")
    print("  Peak-like summaries identify the strongest angular lobe from the orientation trace.")
    print("  Moment-like summaries compute a circular weighted-average direction after subtracting a low angular baseline.")
    print("  One peak candidate keeps the legacy absolute-angle calculation; the rest operate directly on the bead-relative trace.")
    print("  Panel titles report linear-fit R^2 between bead direction and candidate direction.")
    print("  Here |Δθ| = |wrapped(candidate direction - bead direction)| for each cyst.")
    print("  The reported median |Δθ| is the median of that absolute angular error across responsive cysts; lower means closer alignment.")
    print("  Median |Δθ| is shown only as a diagnostic comparison metric and is not used in the final plot below.")

    def _plot_orientation_candidate_group(specs, group_title, ncols=3, figsize=(15.6, 9.0)):
        n = len(specs)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=True, sharey=True)
        axes = np.atleast_1d(axes).ravel()

        x_vals = candidate_df["theta_b_deg"].to_numpy(dtype=float)
        x_vals = x_vals[np.isfinite(x_vals)]
        if len(x_vals):
            x_pad = max(5.0, 0.05 * float(np.nanmax(x_vals) - np.nanmin(x_vals)))
            xlim = (float(np.nanmin(x_vals)) - x_pad, float(np.nanmax(x_vals)) + x_pad)
        else:
            xlim = (-180.0, 180.0)

        y_pool = []
        for col, _ in specs:
            vals = candidate_df[col].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if len(vals):
                y_pool.append(vals)
        if y_pool:
            y_all = np.concatenate(y_pool)
            y_pad = max(5.0, 0.08 * float(np.nanmax(y_all) - np.nanmin(y_all)))
            ylim = (float(np.nanmin(y_all)) - y_pad, float(np.nanmax(y_all)) + y_pad)
        else:
            ylim = (-180.0, 180.0)

        line_x = np.linspace(xlim[0], xlim[1], 400)
        for ax, (col, label) in zip(axes, specs):
            sub = candidate_df[["theta_b_deg", col]].dropna().copy()
            x = sub["theta_b_deg"].to_numpy(dtype=float)
            y = sub[col].to_numpy(dtype=float)
            slope, intercept, r2 = linear_fit_r2(x, y)
            med_err = _median_abs_angle_error_deg(x, y)
            ax.scatter(x, y, s=22, color="tab:blue", alpha=0.78, edgecolors="none")
            if np.isfinite(slope) and np.isfinite(intercept):
                ax.plot(line_x, slope * line_x + intercept, color="red", linestyle="--", linewidth=1.7, label="fit")
            ax.set_title(f"{label}\nR^2={r2:.3f} | median |Δθ|={med_err:.1f}°", fontsize=10)
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)
            ax.set_xlabel("Bead direction (deg)")
            ax.set_ylabel("Candidate direction summary (deg)")
        for ax in axes[n:]:
            ax.axis("off")
        fig.suptitle(group_title, fontsize=12, y=1.01)
        plt.tight_layout()
        plt.show()

    if not RENDER_SENSITIVITY_ANALYSIS_FIGURES:
        print("Skipping Orientation Plot 2 candidate figures (RENDER_SENSITIVITY_ANALYSIS_FIGURES=False).")
    else:
        _plot_orientation_candidate_group(
            peak_specs,
            "Orientation Plot 2 candidate peak-like direction summaries",
            ncols=4,
            figsize=(18.8, 9.2),
        )
        _plot_orientation_candidate_group(
            moment_specs,
            "Orientation Plot 2 candidate moment-like direction summaries",
            ncols=3,
            figsize=(15.6, 9.0),
        )


## Final Plots

Generate the final distance and orientation plots after the quality-control and sensitivity analyses above look acceptable.


In [ ]:
# -------------------------------
# Main traces and regression plots (run after diagnostics look good)
# -------------------------------
bead_summary = summary_df[summary_df["bead_present"].astype(bool)].copy()
control_summary = summary_df[~summary_df["bead_present"].astype(bool)].copy()

if len(bead_summary) == 0:
    raise RuntimeError("No bead-present rows for plotting.")

if len(distance_trace_df) == 0:
    raise RuntimeError("Distance trace table is empty.")
if len(orientation_trace_df) == 0:
    raise RuntimeError("Orientation trace table is empty.")

# Aggregate across cyst-level traces, not across raw pixels.
distance_trace_agg = aggregate_trace_across_cysts(
    distance_trace_df,
    x_col="distance_local_um",
    y_col="ratio_mean",
)

# Compute fit summaries even when plots are suppressed during optimization.
x = bead_summary["distance_d_um"].to_numpy(dtype=float)
y = bead_summary[PLOT2_STANDARD_METRIC_COL].to_numpy(dtype=float)
plot2_regime = _fit_plot2_responsive_regime(
    x,
    y,
    min_points=PLOT2_RESPONSIVE_MIN_POINTS,
    min_fraction=PLOT2_RESPONSIVE_MIN_FRACTION,
)
if plot2_regime.get("valid", False):
    distance_r2 = float(plot2_regime["responsive_r2"])
    distance_cutoff_um = float(plot2_regime["cutoff_um"])
    distance_rho = float(plot2_regime["responsive_rho"]) if np.isfinite(plot2_regime["responsive_rho"]) else np.nan
    distance_plateau_r2 = float(plot2_regime["plateau_linear_r2"]) if np.isfinite(plot2_regime["plateau_linear_r2"]) else np.nan
    distance_plateau_rho = float(plot2_regime["plateau_linear_rho"]) if np.isfinite(plot2_regime["plateau_linear_rho"]) else np.nan
    distance_plateau_p_flat = float(plot2_regime["plateau_flat_p"]) if np.isfinite(plot2_regime["plateau_flat_p"]) else np.nan
else:
    distance_slope, distance_intercept, distance_r2 = linear_fit_r2(x, y)
    distance_cutoff_um = np.nan
    distance_rho = pd.Series(x).corr(pd.Series(y), method="spearman")
    distance_plateau_r2 = np.nan
    distance_plateau_rho = np.nan
    distance_plateau_p_flat = np.nan

if plot2_regime.get("valid", False):
    orientation_plot1_ids = set(
        bead_summary.loc[bead_summary["distance_d_um"] <= distance_cutoff_um, "cyst_id"].dropna().astype(str)
    )
    orientation_plot1_gate_label = f"bead-present cysts with d <= {distance_cutoff_um:.1f} um"
else:
    orientation_plot1_ids = set(bead_summary["cyst_id"].dropna().astype(str))
    orientation_plot1_gate_label = "all bead-present cysts (no stable responsive cutoff)"

orientation_plot1_trace_df = orientation_trace_df[
    orientation_trace_df["cyst_id"].astype(str).isin(orientation_plot1_ids)
].copy()
orientation_plot1_agg = aggregate_trace_across_cysts(
    orientation_plot1_trace_df,
    x_col="theta_rel_bead_deg",
    y_col="ratio_norm_mean",
)
orientation_plot1_n_cysts = int(orientation_plot1_trace_df["cyst_id"].astype(str).nunique())
orientation_plot1_n_total = int(bead_summary["cyst_id"].astype(str).nunique())

orientation_plot2_summary = bead_summary[
    bead_summary["cyst_id"].astype(str).isin(orientation_plot1_ids)
].copy()
orientation_plot2_n_cysts = int(orientation_plot2_summary["cyst_id"].astype(str).nunique())

ORIENTATION_PLOT2_STANDARD_COL = "theta_moment_deg"
ORIENTATION_PLOT2_STANDARD_LABEL = "Shell-pixel first moment direction (20th-pct baseline)"

# The main Orientation Plot 2 uses the stored theta_moment_deg column from
# summary_df, which is the currently selected shell-pixel first-moment
# direction summary (20th-pct baseline). Alternative direction summaries are
# compared separately in the candidate section above.
x2 = orientation_plot2_summary["theta_b_deg"].to_numpy(dtype=float)
orientation_plot2_y = orientation_plot2_summary[ORIENTATION_PLOT2_STANDARD_COL].to_numpy(dtype=float)
orientation_plot2_slope, orientation_plot2_intercept, orientation_plot2_r2 = linear_fit_r2(x2, orientation_plot2_y)

print("Distance Plot 2 standard metric:", PLOT2_STANDARD_METRIC_COL, "|", PLOT2_STANDARD_METRIC_LABEL)
if plot2_regime.get("valid", False):
    print("Distance Plot 2 responsive cutoff (um):", f"{distance_cutoff_um:.1f}")
    print("Distance Plot 2 responsive section R^2 / rho:", f"{distance_r2:.3f}", "/", f"{distance_rho:.3f}")
    print("Distance Plot 2 plateau nested p(flat vs line):", f"{distance_plateau_p_flat:.3g}")
else:
    print("Distance Plot 2 responsive-regime fit unavailable; fallback whole-range R^2:", f"{distance_r2:.3f}")
print("Orientation Plot 1 gate:", orientation_plot1_gate_label)
print("Orientation Plot 1 included cysts:", orientation_plot1_n_cysts, "/", orientation_plot1_n_total)
print("Orientation Plot 2 gate:", orientation_plot1_gate_label)
print("Orientation Plot 2 included cysts:", orientation_plot2_n_cysts, "/", orientation_plot1_n_total)
print("Orientation Plot 2 standard direction summary:", ORIENTATION_PLOT2_STANDARD_LABEL)
print("Orientation Plot 2 fit R^2:", f"{orientation_plot2_r2:.3f}")

if OPTIMIZATION_MODE:
    print("Skipping manuscript-style plots and figure export (OPTIMIZATION_MODE=True).")
else:
    def _save_figure_variants(fig, out_dir, stem):
        stem_dir = out_dir / stem
        stem_dir.mkdir(parents=True, exist_ok=True)
        for ext in FIG_EXPORT_FORMATS:
            out_path = stem_dir / f"{stem}.{ext}"
            save_kwargs = {"bbox_inches": "tight"}
            if ext == "png":
                save_kwargs["dpi"] = FIG_EXPORT_DPI
            fig.savefig(out_path, **save_kwargs)

    FIG1E_PLOT1_STEM = "Fig1e_distance_plot1_local_bead_distance_trace"
    FIG1E_PLOT1_ADAPTIVE_STEM = "Fig1e_distance_plot1_alternative_adaptive_support_trace"
    FIG1E_PLOT2_STEM = "Fig1e_distance_plot2_bead_proximal_response"
    FIG1F_PLOT1_STEM = "Fig1f_orientation_plot1_bead_aligned_trace"
    FIG1F_PLOT2_STEM = "Fig1f_orientation_plot2_first_moment_direction"

    if "_merge_adjacent_exact_distance_bins_by_pixel_support" not in globals():
        def _merge_adjacent_exact_distance_bins_by_pixel_support(bin_df, target_pixels, drop_final_under_target=False):
            sub = bin_df[["distance_local_um", "ratio_mean", "n_pixels"]].dropna().copy()
            if len(sub) == 0:
                return pd.DataFrame(columns=["x_lo", "x_hi", "width_um", "x_center", "ratio_mean", "n_pixels", "n_exact_bins"])
            sub = sub.sort_values("distance_local_um").reset_index(drop=True)
            centers = sub["distance_local_um"].to_numpy(dtype=float)
            values = sub["ratio_mean"].to_numpy(dtype=float)
            weights = sub["n_pixels"].to_numpy(dtype=float)
            positive_weights = weights[np.isfinite(weights) & (weights > 0)]
            if not np.isfinite(target_pixels) or target_pixels <= 0:
                target_pixels = float(np.nanmedian(positive_weights)) if len(positive_weights) else 1.0
            target_pixels = max(float(target_pixels), 1.0)
            rows = []
            i = 0
            half_bin = 0.5 * float(DISTANCE_TRACE_BIN_UM)
            while i < len(sub):
                j = i
                acc = 0.0
                while j < len(sub) and acc < target_pixels:
                    acc += float(weights[j])
                    j += 1
                xs = centers[i:j]
                ys = values[i:j]
                ws = weights[i:j]
                is_final_chunk = bool(j >= len(sub))
                if drop_final_under_target and is_final_chunk and float(np.sum(ws)) < target_pixels:
                    break
                rows.append(
                    {
                        "x_lo": float(xs[0] - half_bin),
                        "x_hi": float(xs[-1] + half_bin),
                        "width_um": float(xs[-1] - xs[0] + float(DISTANCE_TRACE_BIN_UM)),
                        "x_center": float(np.average(xs, weights=ws)) if np.sum(ws) > 0 else float(np.nanmean(xs)),
                        "ratio_mean": float(np.average(ys, weights=ws)) if np.sum(ws) > 0 else float(np.nanmean(ys)),
                        "n_pixels": float(np.sum(ws)),
                        "n_exact_bins": int(len(xs)),
                    }
                )
                i = j
            return pd.DataFrame(rows)

    distance_plot1_bead_df = distance_trace_df.copy()
    if "bead_present" in distance_plot1_bead_df.columns:
        distance_plot1_bead_df = distance_plot1_bead_df.loc[
            distance_plot1_bead_df["bead_present"].astype(bool)
        ].copy()

    pooled_exact_rows = []
    for distance_local_um, sub in distance_plot1_bead_df.groupby("distance_local_um", sort=True):
        weights = sub["n_pixels"].to_numpy(dtype=float)
        pooled_exact_rows.append(
            {
                "distance_local_um": float(distance_local_um),
                "ratio_mean": float(np.average(sub["ratio_mean"].to_numpy(dtype=float), weights=weights)),
                "n_pixels": float(np.sum(weights)),
            }
        )
    distance_plot1_pooled_exact = pd.DataFrame(pooled_exact_rows)
    distance_plot1_pooled_target_pixels = float(np.nanmedian(distance_plot1_pooled_exact["n_pixels"].to_numpy(dtype=float)))
    distance_plot1_pooled_adaptive = _merge_adjacent_exact_distance_bins_by_pixel_support(
        distance_plot1_pooled_exact[["distance_local_um", "ratio_mean", "n_pixels"]],
        distance_plot1_pooled_target_pixels,
        drop_final_under_target=True,
    )

    half_bin = 0.5 * float(DISTANCE_TRACE_BIN_UM)
    adaptive_interval_rows = []
    for _, row in distance_plot1_pooled_adaptive.iterrows():
        lo_center = float(row["x_lo"] + half_bin)
        hi_center = float(row["x_hi"] - half_bin)
        interval_sub = distance_plot1_bead_df.loc[
            (distance_plot1_bead_df["distance_local_um"] >= lo_center - 1e-9)
            & (distance_plot1_bead_df["distance_local_um"] <= hi_center + 1e-9)
        ].copy()
        cyst_interval_vals = []
        for _, cyst_sub in interval_sub.groupby("cyst_id", sort=False):
            weights = cyst_sub["n_pixels"].to_numpy(dtype=float)
            cyst_interval_vals.append(
                float(np.average(cyst_sub["ratio_mean"].to_numpy(dtype=float), weights=weights))
            )
        vals = np.asarray(cyst_interval_vals, dtype=float)
        n = int(np.sum(np.isfinite(vals)))
        mean = float(np.nanmean(vals)) if n else np.nan
        sd = float(np.nanstd(vals, ddof=1)) if n > 1 else (0.0 if n == 1 else np.nan)
        sem = float(sd / np.sqrt(n)) if n > 1 else (0.0 if n == 1 else np.nan)
        adaptive_interval_rows.append(
            {
                "x_lo": float(row["x_lo"]),
                "x_hi": float(row["x_hi"]),
                "x_mid": float(0.5 * (row["x_lo"] + row["x_hi"])),
                "mean": mean,
                "sd": sd,
                "sem": sem,
                "n_cysts": n,
                "n_pixels": float(row["n_pixels"]),
                "n_exact_bins": int(row["n_exact_bins"]),
            }
        )
    distance_plot1_adaptive_interval_agg = pd.DataFrame(adaptive_interval_rows)

    # Distance Plot 1
    fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
    for _, sub in distance_trace_df.groupby("cyst_id"):
        sub = sub.sort_values("distance_local_um")
        ax.plot(
            sub["distance_local_um"],
            sub["ratio_mean"],
            color="0.65",
            alpha=0.14,
            linewidth=0.8,
        )
        ax.scatter(
            sub["distance_local_um"],
            sub["ratio_mean"],
            color="0.65",
            alpha=0.09,
            s=6,
        )
    if len(distance_trace_agg) > 0:
        ax.plot(
            distance_trace_agg["distance_local_um"],
            distance_trace_agg["mean"],
            color="tab:blue",
            linewidth=2.6,
            label="Mean trace",
        )
        ax.fill_between(
            distance_trace_agg["distance_local_um"],
            distance_trace_agg["mean"] - distance_trace_agg["sd"],
            distance_trace_agg["mean"] + distance_trace_agg["sd"],
            color="tab:blue",
            alpha=0.22,
            label="±SD",
        )
    ax.set_xlabel("Local bead-to-shell-pixel distance (um)")
    ax.set_ylabel("Normalized pSMAD/DAPI")
    ax.set_title("Distance Plot 1: Shell pSMAD/DAPI by local bead distance")
    ax.legend(loc="best")
    plt.tight_layout()
    if SAVE_FIGURES:
        _save_figure_variants(fig, FIG1E_DIR, FIG1E_PLOT1_STEM)
    plt.show()

    # Distance Plot 1 alternative: adaptive pooled-support bins
    fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
    for _, sub in distance_plot1_bead_df.groupby("cyst_id"):
        sub = sub.sort_values("distance_local_um")
        ax.plot(
            sub["distance_local_um"],
            sub["ratio_mean"],
            color="0.65",
            alpha=0.14,
            linewidth=0.8,
        )
        ax.scatter(
            sub["distance_local_um"],
            sub["ratio_mean"],
            color="0.65",
            alpha=0.09,
            s=6,
        )
    if len(distance_plot1_adaptive_interval_agg) > 0:
        ax.plot(
            distance_plot1_adaptive_interval_agg["x_mid"],
            distance_plot1_adaptive_interval_agg["mean"],
            color="tab:blue",
            linewidth=2.6,
            label="Mean trace",
        )
        ax.fill_between(
            distance_plot1_adaptive_interval_agg["x_mid"],
            distance_plot1_adaptive_interval_agg["mean"] - distance_plot1_adaptive_interval_agg["sd"],
            distance_plot1_adaptive_interval_agg["mean"] + distance_plot1_adaptive_interval_agg["sd"],
            color="tab:blue",
            alpha=0.22,
            label="±SD",
        )
    ax.set_xlabel("Local bead-to-shell-pixel distance (um)")
    ax.set_ylabel("Normalized pSMAD/DAPI")
    ax.set_title("Distance Plot 1 alternative: adaptive pooled-support bins")
    ax.legend(loc="best")
    if len(distance_plot1_adaptive_interval_agg) > 0:
        alt_ymin = float(np.nanmin(distance_plot1_adaptive_interval_agg["mean"] - distance_plot1_adaptive_interval_agg["sd"]))
        alt_ymax = float(np.nanmax(distance_plot1_adaptive_interval_agg["mean"] + distance_plot1_adaptive_interval_agg["sd"]))
        alt_span = max(1e-6, alt_ymax - alt_ymin)
        ax.set_ylim(alt_ymin - 0.06 * alt_span, alt_ymax + 0.02 * alt_span)
    if SAVE_FIGURES:
        _save_figure_variants(fig, FIG1E_DIR, FIG1E_PLOT1_ADAPTIVE_STEM)
    plt.tight_layout()
    plt.show()

    # Distance Plot 2
    fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
    bead_plot2 = bead_summary[["distance_d_um", PLOT2_STANDARD_METRIC_COL]].dropna().copy()
    bx = bead_plot2["distance_d_um"].to_numpy(dtype=float)
    by = bead_plot2[PLOT2_STANDARD_METRIC_COL].to_numpy(dtype=float)
    ax.scatter(bx, by, s=30, alpha=0.78, color="tab:blue", label="Bead-present cysts")

    if plot2_regime.get("valid", False):
        xx_left = np.linspace(float(plot2_regime["x_left_min"]), float(plot2_regime["cutoff_um"]), 100)
        yy_left = plot2_regime["responsive_slope"] * xx_left + plot2_regime["responsive_intercept"]
        ax.plot(
            xx_left,
            yy_left,
            color="crimson",
            linestyle="--",
            linewidth=1.8,
            label="Responsive fit",
        )
        ax.axvline(
            float(plot2_regime["cutoff_um"]),
            color="crimson",
            linestyle=":",
            linewidth=1.3,
            label=f"Responsive cutoff ({plot2_regime['cutoff_um']:.0f} um)",
        )
        ax.hlines(
            float(plot2_regime["plateau_mean"]),
            float(plot2_regime["cutoff_um"]),
            float(plot2_regime["x_all_max"]),
            colors="tab:orange",
            linestyles="-.",
            linewidth=1.5,
            label="Distal plateau",
        )
        plateau_p_txt = f"{plot2_regime['plateau_flat_p']:.2g}" if np.isfinite(plot2_regime['plateau_flat_p']) else "NA"
        stats_txt = (
            f"Responsive regime: n={plot2_regime['responsive_n']}, R^2={plot2_regime['responsive_r2']:.3f}\n"
            f"Distal regime: n={plot2_regime['plateau_n']}, flat-vs-line p={plateau_p_txt}\n"
            f"Cutoff = {plot2_regime['cutoff_um']:.0f} um"
        )
        ax.text(
            0.03,
            0.97,
            stats_txt,
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.86, edgecolor="0.75"),
        )
    else:
        distance_slope, distance_intercept, _ = linear_fit_r2(bx, by)
        if np.isfinite(distance_slope):
            xx = np.linspace(float(np.nanmin(bx)), float(np.nanmax(bx)), 200)
            yy = distance_slope * xx + distance_intercept
            ax.plot(xx, yy, linewidth=2.0, color="crimson", linestyle="--", label=f"Linear fit (R^2={distance_r2:.3f})")

    if PLOT2_STANDARD_SHOW_CONTROLS and PLOT2_STANDARD_METRIC_COL in control_summary.columns:
        control_plot2 = control_summary[["distance_plot_um", PLOT2_STANDARD_METRIC_COL]].dropna().copy()
        if len(control_plot2) > 0:
            ax.scatter(
                control_plot2["distance_plot_um"],
                control_plot2[PLOT2_STANDARD_METRIC_COL],
                marker="s",
                s=44,
                alpha=0.90,
                color="0.35",
                label="No-bead controls",
            )

    ax.set_xlabel("Nearest bead-center to cyst-surface distance d (um)")
    ax.set_ylabel(PLOT2_STANDARD_METRIC_LABEL)
    ax.set_title("Distance Plot 2: Bead-proximal shell response vs bead distance")
    ax.legend(loc="best")
    plt.tight_layout()
    if SAVE_FIGURES:
        _save_figure_variants(fig, FIG1E_DIR, FIG1E_PLOT2_STEM)
    plt.show()

    # Orientation Plot 1
    fig = plt.figure(figsize=(7.4, 5.6))
    for _, sub in orientation_plot1_trace_df.groupby("cyst_id"):
        sub = sub.sort_values("theta_rel_bead_deg")
        plt.plot(
            sub["theta_rel_bead_deg"],
            sub["ratio_norm_mean"],
            color="0.65",
            alpha=0.14,
            linewidth=0.8,
        )
        plt.scatter(
            sub["theta_rel_bead_deg"],
            sub["ratio_norm_mean"],
            color="0.65",
            alpha=0.09,
            s=6,
        )
    if len(orientation_plot1_agg) > 0:
        plt.plot(
            orientation_plot1_agg["theta_rel_bead_deg"],
            orientation_plot1_agg["mean"],
            color="tab:green",
            linewidth=2.6,
            label="Mean trace",
        )
        plt.fill_between(
            orientation_plot1_agg["theta_rel_bead_deg"],
            orientation_plot1_agg["mean"] - orientation_plot1_agg["sd"],
            orientation_plot1_agg["mean"] + orientation_plot1_agg["sd"],
            color="tab:green",
            alpha=0.22,
            label="±SD",
        )
    plt.axvline(0.0, color="red", linestyle="--", linewidth=1.5, label="Bead direction")
    plt.title("Orientation Plot 1: Angular pSMAD traces aligned to bead direction")
    plt.xlabel("Angle relative to bead direction (deg)")
    plt.ylabel("Within-cyst normalized pSMAD/DAPI")
    plt.legend(loc="best")
    plt.tight_layout()
    if SAVE_FIGURES:
        _save_figure_variants(fig, FIG1F_DIR, FIG1F_PLOT1_STEM)
    plt.show()

    # Orientation Plot 2
    fig = plt.figure(figsize=(6.8, 5.6))
    plt.scatter(x2, orientation_plot2_y, s=30, alpha=0.80, color="tab:blue")
    if np.isfinite(orientation_plot2_slope):
        xx2 = np.linspace(float(np.nanmin(x2)), float(np.nanmax(x2)), 200)
        yy2 = orientation_plot2_slope * xx2 + orientation_plot2_intercept
        plt.plot(xx2, yy2, linewidth=2.2, color="crimson", linestyle="--", label=f"Linear fit (R^2={orientation_plot2_r2:.3f})")
    plt.xlabel("Bead direction (deg)")
    plt.ylabel("First-moment pSMAD direction (deg)")
    plt.title("Orientation Plot 2: pSMAD first-moment direction vs bead direction")
    plt.legend(loc="best")
    plt.tight_layout()
    if SAVE_FIGURES:
        _save_figure_variants(fig, FIG1F_DIR, FIG1F_PLOT2_STEM)
    plt.show()


## Representative Coordinate Diagrams

Use one clean bead-present array to show the exact coordinate definitions behind the distance and orientation analyses. Distance and orientation start from the same shell pixels; the distance row emphasizes `d` and the exact `10 um` local-distance bins, while the orientation row emphasizes `theta_b` and the exact `5 deg` bead-relative angle bins.


In [ ]:
# -------------------------------
# Representative coordinate diagrams
# -------------------------------
REPRESENTATIVE_COORDINATE_CYST_ID = "manual_s13_cyst_0001"
DISTANCE_COORD_DIAGRAM_STEM = "distance_coord_diagram"
ORIENTATION_COORD_DIAGRAM_STEM = "orientation_coord_diagram"

if OPTIMIZATION_MODE:
    print("Skipping representative coordinate diagrams (OPTIMIZATION_MODE=True).")
else:
    print("Representative coordinate-diagram cyst:", REPRESENTATIVE_COORDINATE_CYST_ID)

    def _save_coord_diagram_variants(fig, out_dir, stem):
        stem_dir = out_dir / stem
        stem_dir.mkdir(parents=True, exist_ok=True)
        for ext in FIG_EXPORT_FORMATS:
            save_kwargs = {"dpi": FIG_EXPORT_DPI, "bbox_inches": "tight"}
            if str(ext).lower() != "png":
                save_kwargs.pop("dpi", None)
            out_path = stem_dir / f"{stem}.{ext}"
            fig.savefig(out_path, **save_kwargs)
            print("Saved representative coordinate diagram:", out_path)

    fig_distance = plot_representative_distance_coordinate_diagram(
        REPRESENTATIVE_COORDINATE_CYST_ID
    )
    _save_coord_diagram_variants(fig_distance, FIG1E_DIR, DISTANCE_COORD_DIAGRAM_STEM)

    fig_orientation = plot_representative_orientation_coordinate_diagram(
        REPRESENTATIVE_COORDINATE_CYST_ID
    )
    _save_coord_diagram_variants(
        fig_orientation, FIG1F_DIR, ORIENTATION_COORD_DIAGRAM_STEM
    )
